#### Revised from https://www.kaggle.com/code/nihilisticneuralnet/baseline-nvarc-arc-25-winning-solution-for-t4x2?scriptVersionId=307086709 (version 1)

### 결합 노트북 v2.4.2 — kgmon + 우리 스위프

**전략 (확정 원칙)**: ①우리 스위프 먼저 — 싸게, 몫 고정(rerun 1.67h: U 1200/색칠 600/계열 3000/항등 1200s)
→ ②kgmon 이 남는 시간 전부 (attempt_1 불가침) → ③병합은 빈 유닛 채움 + attempt_2 만.
근거: 2step V4 27.92 회귀 = 본대 시간을 뺏으면 역효과.

**이번 판 변경**
- 스위프 v2.4: 색칠(shapecolor) **토큰레벨 패치** — 지난 커밋의 `WordLevel UNK` 전멸 수정 (단독 코랩 스모크에서 rc=0·N>0 검증됨)
- [F위반 측정 셀] 신규 삽입 (kgmon 제출 다음, 병합 이전 · 오프라인 전용)
- 병합 v2.4.2: `FORCE_FIX_F=True(0.20)` / `SIL_TIERS=(0,)` / `KGMON_F_REPAIR=False` + 3색도형 오폐기 수정

**Add Input (4종)**: `sorokin/qwen3_4b_grids15_sft139` · `kwonyongdeuk/arc2-shapeuni`(shu4val) ·
`kwonyongdeuk/arc2-adapters-v2` · `kwonyongdeuk/arc2-shapecolor`

**읽을 로그 (오프라인 커밋)**
1. `[색칠] 페이즈 종료 rc=0 (…, N쌍둥이)` — N>0 (색칠 부활 확인)
2. `[적중]` 태그 분해 — `C…` 최초 관측 여부
3. [F위반 측정] `KILLED==0 & RESCUED>0` 이면 다음 제출에서 `KGMON_F_REPAIR=True`
4. `병합 후` 오프라인 점수 (테스트 커밋 기준선 7.67 · kgmon 4태스크 한정)

실전 12h 커밋 전 `TEST_KGMON_TASKS = None` 확인 (스위프 셀 상단).

In [1]:
import time, json, os
global_end_time = time.time() + 12 * 3600 - 600

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    _p = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    with open(_p) as f: _t = json.load(f)
    _d = {k: [{"attempt_1": [[0]], "attempt_2": [[0]]}
              for _ in range(len(_t[k]["test"]))] for k in _t}
    with open("submission.json", "w") as f: json.dump(_d, f)
    print(f"dummy submission.json ({len(_d)} tasks)")

In [2]:
!pip uninstall -y tensorflow

Found existing installation: tensorflow 2.18.0
Uninstalling tensorflow-2.18.0:
  Successfully uninstalled tensorflow-2.18.0


In [3]:
%%writefile arc_loader.py
import json
import numpy as np
from transformers import AutoTokenizer


def convert_grid_to_string(grid) -> str:
    text = ""
    for row in grid:
        for cell in row:
            text += str(int(cell))
        text += "\n"
    return text.strip()

def is_valid_solution(guess):
    return isinstance(guess, np.ndarray) and guess.ndim == 2 and all(0 < x <= 30 for x in guess.shape)

def shuffled(data_list):
    return np.random.permutation(data_list).tolist()

def permute_mod(a, descriptor, invert=False):
    permutation = [int(i) for i in descriptor if str(i).isdigit()]
    assert sorted(permutation)==list(range(10))
    a = np.asarray(a)
    if a.ndim==3:
        if not invert: permutation = np.argsort(permutation)
        a = a[..., permutation]
    else:
        assert a.ndim==2
        if invert: permutation = np.argsort(permutation)
        a = np.asarray(permutation)[a]
    return a

def permute_rnd_all_(query):
    permutation = np.random.permutation(10).tolist()
    return 'permute' + ''.join(map(str, permutation))


class QwenFormatter:

    def __init__(self, tokenizer: AutoTokenizer):
        self.tokenizer = tokenizer

    def fmt_query(self, query) -> str:
        grid_input = convert_grid_to_string(query[0]["input"])
        return "<|im_start|>user\n" + grid_input + "<|im_end|><|im_start|>assistant\n"

    def fmt_reply(self, reply) -> str:
        return convert_grid_to_string(reply[0]) + "<|im_end|>"

    def fmt_train(self, train, last_is_challenge=False) -> str:
        if last_is_challenge:
            test = train[-1]
            train = train[:-1]
        else:
            test = None
        text = ""
        for x in train:
            grid_input = convert_grid_to_string(x["input"])
            grid_output = convert_grid_to_string(x["output"])
            text += f"<|im_start|>user\n{grid_input}<|im_end|><|im_start|>assistant\n{grid_output}<|im_end|>"
        if test is not None:
            text += self.fmt_query([test]) + self.fmt_reply([test["output"]])
        return text

    def max_new_tokens(self):
        max_sized_reply = np.zeros([30, 30], dtype=int)
        tokens = self.tokenizer.encode(self.fmt_reply([max_sized_reply]))
        return len(tokens) + 1

    def convert_tokens_to_array(self, tokens, limit_rows=30):
        if len(tokens) < 2:
            return None
        text = self.tokenizer.decode(tokens[:-1])
        try:
            lines = text.strip().split("\n")
            by_rows = [row for row in [[int(x) for x in line if x.isdigit()] for line in lines] if len(row)]
            if len(by_rows) > limit_rows:
                by_rows = by_rows[:limit_rows]
            array = np.array(by_rows, dtype=int)
            if is_valid_solution(array):
                return array
        except:
            pass
        return None


class ArcDataset:

    @staticmethod
    def forward_mod(a, key, use_perm=True):
        if a is None: return a
        for op in key.split('.')[1:]:
            if   op=='rot90':              a = np.rot90(a)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=False) if use_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    @staticmethod
    def invert_mod(a, key, inv_perm=True):
        if a is None: return a
        for op in key.split('.')[1:][::-1]:
            if   op=='rot90':              a = np.rot90(a, k=3)
            elif op=='transpose':          a = np.swapaxes(a, 0, 1)
            elif op.startswith('permute'): a = permute_mod(a, op, invert=True) if inv_perm else a
            elif op.startswith('copy'):    a = np.copy(a)
            elif op.startswith('out'):     a = a
            elif op.startswith('ex'):      a = a
            elif op.startswith('run'):     a = a
            else: raise NotImplementedError(f"Inversion of operation '{op}' unknown.")
        return a

    def __init__(self, queries, replies={}, keys=None, is_orig=False):
        if keys is not None: keys = [k for k in keys if k is not None]
        self.queries = queries if keys is None else {k: queries[k] for k in keys}
        self.replies = replies if keys is None else {k: replies[k] for k in keys if k in replies}
        self.is_orig = is_orig
        self.keys = sorted(queries.keys()) if keys is None else keys
        self.transposed_dataset = None

    def change_keys(self, keys, keep_flags=False):
        flags = dict(is_orig=self.is_orig) if keep_flags else {}
        return self.__class__(queries=self.queries, replies=self.replies, keys=keys, **flags)

    @classmethod
    def from_file(cls, queries_file, keys=None):
        with open(queries_file) as f:
            queries = f.read()
        return cls(
            queries=json.loads(queries),
            is_orig=True,
            keys=keys,
        )

    def load_replies(self, replies_file):
        print(f"*** Load solutions from '{replies_file}'...")
        with open(replies_file) as f: replies = f.read()
        replies_parsed = json.loads(replies)
        self.replies = {k: replies_parsed[k] for k in self.keys}
        return self

    def split_multi_replies(self):
        key_indices = [(k, i) for k in self.keys for i in range(len(self.queries[k]['test']))]
        return self.__class__(
            keys=[f'{k}_{i}' for k, i in key_indices],
            queries={f'{k}_{i}': {'train': self.queries[k]['train'], 'test': [self.queries[k]['test'][i]]} for k, i in key_indices},
            replies={f'{k}_{i}': [self.replies[k][i]] for k, i in key_indices if k in self.replies},
        )

    def shuffled(self):
        return self.__class__(queries=self.queries, replies=self.replies, keys=shuffled(self.keys))

    def append(*datasets):
        return datasets[0].__class__(
            queries={k: v for d in datasets for k, v in d.queries.items()},
            replies={k: v for d in datasets for k, v in d.replies.items()},
            keys   =[k    for d in datasets for k    in d.keys           ],
        )

    def mod_single(self, mod_func, descriptor, i, keep_key, inputs_only):
        queries = {}
        replies = {}
        keys    = []
        for k0 in self.keys:
            desc = (('copy{i}' if mod_func is np.copy else mod_func.__name__) if descriptor is None else descriptor if isinstance(descriptor, str) else descriptor(self.queries[k0])).format(i=i)
            func = lambda a, d: np.asarray(mod_func(a) if descriptor is None else mod_func(a, d)).tolist()
            k1 = k0 if keep_key else f"{k0}.{'I' if inputs_only else ''}{desc}"
            keys.append(k1)
            queries[k1] = {m: [{t: (func(a, desc) if t=='input' or not inputs_only else a) for t, a in x.items()} for x in e] for m, e in self.queries[k0].items()}
            if k0 in self.replies:
                replies[k1] = [func(a, desc) for a in self.replies[k0]]
        ret = self.__class__(queries=queries, replies=replies, keys=keys)
        return ret

    def mod(self, mod_func, descriptor=None, n=1, stack=None, keep=False, keep_key=False, shuffle=False, join=True, inputs_only=False):
        assert not (keep and keep_key)
        cur = self
        ret = [cur.shuffled() if shuffle else cur] if keep else []
        if stack is None: stack = mod_func.__name__.startswith('rot')
        for i in range(n):
            cur = (cur if stack else self).mod_single(mod_func, descriptor, i=i, keep_key=keep_key, inputs_only=inputs_only)
            ret.append(cur.shuffled() if shuffle else cur)
        return self.__class__.append(*ret) if join else ret

    def get(self, key, formatter: QwenFormatter):
        train = formatter.fmt_train(self.queries[key]['train'])
        query = formatter.fmt_query(self.queries[key]['test'])
        reply = formatter.fmt_reply(self.replies[key]) if key in self.replies else ''
        text = train+query+reply if reply else formatter.fmt_train(self.queries[key]['train'], last_is_challenge=True)
        return dict(key=key, train=train, query=query, reply=reply, input=train+query, text=text)

    def as_list(self, formatter: QwenFormatter):
        return [self.get(key, formatter) for key in self.keys]

    def get_length(self, key, formatter: QwenFormatter, name, max_of_transposed=False):
        if formatter is None:
            if   name=='input': return sum(np.prod(np.shape(v)) for v3 in self.queries[key].values() for v2 in v3 for v in v2.values())
            elif name=='reply': return sum(np.prod(np.shape(v)) for v in self.replies[key])
            else: assert False
        else:
            datasets = [self]
            if max_of_transposed:
                if self.transposed_dataset is None: self.transposed_dataset = self.mod(np.transpose, keep=False, keep_key=True)
                datasets.append(self.transposed_dataset)
            return max(len(formatter.tokenizer.encode(ds.get(key, formatter=formatter)[name])) for ds in datasets)

    def cut_to_len(self, formatter, name, max_len, from_end=False):
        temp_ds = self.change_keys(self.keys)
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            reply = temp_ds.replies.get(key)
            while max_len<temp_ds.get_length(key, formatter=formatter, name=name):
                query = temp_ds.queries[key]
                if not key.split('.')[-1].startswith('ex'):
                    key = f"{key}.ex{''.join(map(str, range(len(query['train']))))}"
                key_split = key.split('.')
                assert key_split[-1].startswith('ex')
                key = '.'.join(key_split[:-1] + [f'ex{key_split[-1][2:-1] if from_end else key_split[-1][3:]}'])
                temp_ds.queries[key] = {k: ((v[:-1] if from_end else v[1:]) if k=='train' else v) for k, v in query.items()}
                if reply is not None:
                    temp_ds.replies[key] = reply
            new_keys.append(key)
            new_queries[key] = temp_ds.queries[key]
            if reply is not None: new_replies[key] = reply
        return self.__class__(keys=new_keys, queries=new_queries, replies=new_replies)
    
    def shuffle_ex(self, perm=None, keep_max=None):
        new_keys = []
        new_queries = {}
        new_replies = {}
        for key in self.keys:
            n = len(self.queries[key]['train'])
            p = np.random.permutation(n) if perm is None else perm
            if keep_max is not None: p = p[:keep_max]
            new_key = f'{key}.ex' + ('-' if (p.max()>9) else '').join(map(str, p.tolist()))
            new_keys.append(new_key)
            new_queries[new_key] = {k: (np.array(v, dtype=object)[p].tolist() if k=='train' else v) for k, v in self.queries[key].items()}
            if key in self.replies: new_replies[new_key] = self.replies[key]
        return self.__class__(queries=new_queries, replies=new_replies, keys=new_keys)

    def augment(self, n=1, shfl_keys=False, seed=42):
        np.random.seed(seed)
        d = self
        d = d.mod(np.transpose, keep=True)
        d = d.mod(np.rot90, n=3, keep=True)
        d = d.mod(permute_mod, permute_rnd_all_, n=n, shuffle=shfl_keys, keep=False)
        d = d.shuffle_ex()
        return d

    def get_submission(self, results=None):
        assert self.is_orig==True, 'Must be run on original dataset.'
        submission = {k: [{f'attempt_{i+1}': [[0]] for i in range(2)} for _ in range(len(self.queries[k]['test']))] for k in self.keys}
        if results is not None: self.fill_submission(results, submission)
        return submission

    @staticmethod
    def fill_submission(results, submission):
        print(f'*** Generating submission for {len(results)} outputs...')
        for k, v in results.items():
            base_id, base_nr = k.split('_')
            target_dict = submission[base_id][int(base_nr)]
            for i, g in enumerate(v[:len(target_dict)]):
                target_dict[f'attempt_{i+1}'] = g.tolist()

    def validate_submission(self, submission):
        assert self.is_orig==True, 'Must be run on original dataset.'
        score = 0
        for k, v in self.replies.items():
            for i, r in enumerate(v):
                for attempt in ['attempt_1', 'attempt_2']:
                    if np.array_equal(r, submission[k][i][attempt]):
                        score += 1 / len(v)
                        break
        return score

Writing arc_loader.py


In [4]:
%%writefile arc_decoder.py
import os
import bz2
import pickle
import numpy as np

def hashable(guess):
    return tuple(map(tuple, guess))

def score_sum(guesses, getter):
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        x = scores[h] = scores.get(h, [[], g["solution"]])
        x[0].append(g)
    scores = [(getter(sc), o) for sc, o in scores.values()]
    scores = sorted(scores, key=(lambda x: x[0]), reverse=True)
    ordered_outputs = [x[-1] for x in scores]
    return ordered_outputs

def getter_full_probmul_3(guesses, baseline=3):
    inf_score = np.sum([baseline-g["beam_score"] for g in guesses])
    aug_score = np.mean([np.sum([baseline-s for s in g["score_aug"]]) for g in guesses])
    return inf_score + aug_score

def score_full_probmul_3(guesses):
    return score_sum(guesses, getter_full_probmul_3)

def getter_kgmon(guesses):
    inf_score = len(guesses)
    aug_score = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return inf_score - aug_score

def score_kgmon(guesses):
    return score_sum(guesses, getter_kgmon)


def score_entropy_weighted(guesses):
    """Entropy-weighted majority vote. beam_score 낮을수록(=확률 높을수록) 높은 가중치."""
    guess_list = list(guesses.values())
    scores = {}
    for g in guess_list:
        h = hashable(g["solution"])
        w = np.exp(-g["beam_score"])
        if h in scores:
            scores[h][0] += w
        else:
            scores[h] = [w, g["solution"]]
    ranked = sorted(scores.values(), key=lambda x: -x[0])
    return [x[1] for x in ranked]

def getter_dfs_count_x_aug_geomean(guesses):
    """log(dfs_count) - mean_aug_nll. score_aug/beam_score는 NLL이라 음수 부호."""
    dfs_count = len(guesses)
    mean_aug_nll = np.mean([np.mean(g["score_aug"]) for g in guesses])
    return np.log(dfs_count) - mean_aug_nll

def score_dfs_count_x_aug_geomean(guesses):
    return score_sum(guesses, getter_dfs_count_x_aug_geomean)
    

selection_algorithms = [
    score_full_probmul_3,
    score_kgmon,
    score_entropy_weighted,
    score_dfs_count_x_aug_geomean,
]


class ArcDecoder:
    
    def __init__(self, dataset, n_guesses):
        self.dataset = dataset
        self.n_guesses = n_guesses
        self.decoded_results = {}

    def load_decoded_results(self, store, run_name=""):
        for key in os.listdir(store):
            with bz2.BZ2File(os.path.join(store, key)) as f:
                outputs = pickle.load(f)
            base_key = key.split(".")[0]
            self.decoded_results[base_key] = self.decoded_results.get(base_key, {})
            for i, sample in enumerate(outputs):
                self.decoded_results[base_key][f"{key}{run_name}.out{i}"] = sample

    def run_selection_algo(self, selection_algorithm=score_kgmon):
        return {bk: selection_algorithm({k: g for k, g in v.items()}) for bk, v in self.decoded_results.items()}

    def benchmark_selection_algos(self):
        print("*** Benchmark selection algorithms...")

        labels = {}
        num_tasks_per_puzzle = {}
        num_solved_keys = 0
        num_total_keys = 0

        correct_beam_scores = []

        for basekey, basevalues in self.decoded_results.items():

            mult_key, mult_sub = basekey.split("_")
            num_tasks_per_puzzle[mult_key] = max(num_tasks_per_puzzle.get(mult_key, 0), int(mult_sub) + 1)

            labels[basekey] = correct_solution = self.dataset.replies[basekey][0]

            for subkey, sample in basevalues.items():

                solution = sample["solution"]
                beam_score = sample["beam_score"]
                aug_mean = np.mean(sample["score_aug"])

                if np.shape(correct_solution) != np.shape(solution):
                    corr_str = "bad_xy_size"
                elif np.array_equal(correct_solution, solution):
                    corr_str = "ALL_CORRECT"
                    num_solved_keys += 1
                    correct_beam_scores.append(beam_score)
                else:
                    corr_str = "bad_content"

                output_len = f"{solution.shape[0]}x{solution.shape[1]}"

                if corr_str == "ALL_CORRECT":
                    print(f"{corr_str}:{beam_score:8.5f} - {aug_mean:8.5f} {output_len:5s} [{subkey}]")
                num_total_keys += 1

        print(f" subkeys: {num_solved_keys}/{num_total_keys}")
        if len(correct_beam_scores) > 0:
            print(f" avg correct beam score: {np.mean(correct_beam_scores):8.5f}")
            print(f" max correct beam score: {np.max(correct_beam_scores):8.5f}")
        else:
            print(f" no correct beam scores")

        num_puzzles = len(num_tasks_per_puzzle)

        for selection_algorithm in selection_algorithms:
            name = selection_algorithm.__name__
            selected = self.run_selection_algo(selection_algorithm)
            correct_puzzles = {k for k, v in selected.items() if any(np.array_equal(guess, labels[k]) for guess in v[:self.n_guesses])}
            print(correct_puzzles)
            score = sum(1/num_tasks_per_puzzle[k.split("_")[0]] for k in correct_puzzles)
            print(f" acc: {score:5.1f}/{num_puzzles:3} ('{name}')")

Writing arc_decoder.py


In [5]:
%%writefile arc_solver.py
from unsloth import FastLanguageModel, UnslothTrainingArguments, UnslothTrainer
from arc_loader import ArcDataset, QwenFormatter

import gc
import os
import io
import time
import torch
import numpy as np
from tqdm import tqdm
from datasets import Dataset
from collections import defaultdict

from typing import Any, Union
from transformers import DataCollatorForLanguageModeling

import logging
from contextlib import redirect_stdout, redirect_stderr

from peft import get_peft_model_state_dict, set_peft_model_state_dict

import bz2
import pickle

logging.disable(logging.WARNING)

ARC_VOCAB = {
    "0": 0,
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8": 8,
    "9": 9,
    "Ċ": 10,
    "<|im_end|>": 15,
}

ARC_TOKENS = list(ARC_VOCAB.values())
USER_TOKEN_ID = 11
ASSISTANT_TOKEN_ID = 12
PAD_ID = 13
EOS_ID = 15


class UnslothFixedTrainer(UnslothTrainer):

    # Issue https://github.com/unslothai/unsloth/issues/2435

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        """Fixed compute_loss that handles Unsloth's view tensor issue"""
        if self.label_smoother is not None and "labels" in inputs:
            labels = inputs.pop("labels")
        else:
            labels = None
        outputs = model(**inputs)
        if labels is not None:
            unwrapped_model = self.accelerator.unwrap_model(model)
            if hasattr(unwrapped_model, "_get_name") and "unsloth" in unwrapped_model._get_name().lower():
                loss = self.label_smoother(outputs, labels, shift_labels=True)
            else:
                loss = self.label_smoother(outputs, labels)
        else:
            loss = outputs["loss"] if isinstance(outputs, dict) else outputs[0]
        # 🔧 KEY FIX: Clone the loss tensor before in-place operations
        if hasattr(loss, "clone"):
            loss = loss.clone()  # Converts view tensor to independent tensor
        # Now safe for DDP gradient scaling
        if self.accelerator.num_processes > 1:
            loss = loss * self.accelerator.num_processes
        return (loss, outputs) if return_outputs else loss


class QwenDataCollatorForCompletionOnlyLM(DataCollatorForLanguageModeling):

    def torch_call(self, examples: list[Union[list[int], Any, dict[str, Any]]]) -> dict[str, Any]:
        batch = super().torch_call(examples)
        for i in range(len(examples)):
            labels = batch["input_ids"][i].clone()
            user_start_idx = np.where(labels == USER_TOKEN_ID)[0].tolist()
            assistant_start_idx = np.where(labels == ASSISTANT_TOKEN_ID)[0].tolist()
            start_idx = sorted(user_start_idx + assistant_start_idx)
            end_idx = np.where(labels == EOS_ID)[0]
            batch["labels"][i, :] = -100
            for j, (start, end) in enumerate(zip(start_idx, end_idx)):
                assert start < end
                if j % 2 == 1:
                    start += 2
                    end += 1
                    batch["labels"][i, start:end] = labels[start:end]
        return batch


def turbo_dfs(model, logits, max_new_tokens, max_score, scores, pos, cache, start_time, end_time) -> dict:

    n = logits.size(0)

    nll = torch.tensor(scores, dtype=torch.bfloat16).view(n, 1) - logits.float().cpu().log_softmax(-1)

    suffixes = defaultdict(list)

    candidates = dict()

    for i in range(n):
        candidates[i] = []
        for t in ARC_TOKENS:
            score = nll[i, t].item()
            if score < max_score:
                if t == EOS_ID:
                    suffixes[i].append((score, [t]))
                elif max_new_tokens > 1:
                    candidates[i].append((score, t))

    for i in range(n):
        candidates[i] = sorted(candidates[i], key=lambda x:x[0]) #[:5]
    
    while time.time() - start_time < 540 and time.time() < end_time:

        batch_tokens = []
        batch_scores = []
        num_alive_beams = 0

        for i in range(n):
            if len(candidates[i]) == 0:
                batch_tokens.append(PAD_ID)
                batch_scores.append(1000)
            else:
                score, t = candidates[i].pop(0)
                batch_tokens.append(t)
                batch_scores.append(score)
                num_alive_beams += 1

        if num_alive_beams == 0:
            break

        outputs = model(
            input_ids=torch.tensor(batch_tokens, device=model.device, dtype=torch.long).view(-1, 1),
            position_ids=torch.full((n, 1), pos, device=model.device),
            past_key_values=cache,
            return_dict=True,
            use_cache=True,
        )

        next_suffixes = turbo_dfs(
            model,
            logits=outputs.logits[:, -1],
            max_new_tokens=max_new_tokens-1,
            max_score=max_score,
            scores=batch_scores,
            pos=pos+1,
            cache=outputs.past_key_values,
            start_time=start_time,
            end_time=end_time,
        )

        for batch_id, beams in next_suffixes.items():
            for score, suffix_tokens in beams:
                suffix_tokens.insert(0, batch_tokens[batch_id])
                suffixes[batch_id].append((score, suffix_tokens))

    return suffixes


@torch.no_grad()
def inference_turbo_dfs(model, prefix_tokens, max_new_tokens, max_score, end_time):
    input_ids = torch.tensor(prefix_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    suffixes = turbo_dfs(
        model,
        logits=outputs.logits[:, -1],
        max_new_tokens=max_new_tokens,
        max_score=max_score,
        scores=[0.0] * input_ids.size(0),
        pos=input_ids.size(1),
        cache=outputs.past_key_values,
        start_time=time.time(),
        end_time=end_time,
    )
    result = []
    for batch_id, beams in suffixes.items():
        sorted_beams = sorted(beams, key=lambda x:x[0])
        result.append((batch_id, sorted_beams))
    return result


@torch.no_grad()
def calc_scores(queries, answers, tokenizer, model):
    batch_query_tokens = []
    batch_answer_tokens = []
    batch_tokens = []
    batch_lengths = []
    for query, answer in zip(queries, answers):
        query_tokens = tokenizer.encode(query)
        answer_tokens = tokenizer.encode(answer)
        tokens = query_tokens + answer_tokens
        batch_query_tokens.append(query_tokens)
        batch_answer_tokens.append(answer_tokens)
        batch_tokens.append(tokens)
        batch_lengths.append(len(tokens))
    max_len = max(batch_lengths)
    padded_tokens = []
    for tokens in batch_tokens:
        padded = tokens + [PAD_ID] * (max_len - len(tokens))
        padded_tokens.append(padded)
    input_ids = torch.tensor(padded_tokens, device=model.device, dtype=torch.long)
    outputs = model(input_ids=input_ids, return_dict=True, use_cache=True)
    batch_logits = outputs.logits.float().cpu().log_softmax(-1)
    result = []
    for logits, query_tokens, answer_tokens in zip(batch_logits, batch_query_tokens, batch_answer_tokens):
        query_length = len(query_tokens)
        answer_logits = logits[query_length-1:query_length-1+len(answer_tokens)]
        answer_score = answer_logits[torch.arange(len(answer_tokens)), answer_tokens].sum()
        result.append(-answer_score.item())
    return result


def worker(rank, queue, end_time):

    os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

    import torch

    torch.backends.cuda.enable_flash_sdp(True)
    torch.backends.cuda.enable_mem_efficient_sdp(True)
    torch.backends.cuda.enable_math_sdp(False)

    import unsloth.models.qwen3 as qwen3_module
    import torch.nn.functional as F
    
    def patched_attention(Q, K, V, *args, **kwargs):
        # Q,K,V shape: (bsz, seqlen, num_heads, head_dim)
        Q = Q.transpose(1, 2)  # (bsz, num_heads, seqlen, head_dim)
        K = K.transpose(1, 2)
        V = V.transpose(1, 2)
        # Expand K,V for GQA (repeat to match Q's head count)
        num_q_heads = Q.shape[1]
        num_kv_heads = K.shape[1]
        if num_q_heads != num_kv_heads:
            factor = num_q_heads // num_kv_heads
            K = K.repeat_interleave(factor, dim=1)
            V = V.repeat_interleave(factor, dim=1)
        scale = Q.shape[-1] ** -0.5
        scores = torch.matmul(Q, K.transpose(-2, -1)) * scale
        weights = torch.softmax(scores.float(), dim=-1).to(Q.dtype)
        out = torch.matmul(weights, V)
        return out.transpose(1, 2)  # back to (bsz, seqlen, num_heads, head_dim)
    
#    qwen3_module.flash_attn_func = patched_attention
    import torch.cuda.amp as amp
    amp.GradScaler = lambda **kwargs: torch.cuda.amp.GradScaler(enabled=False, **kwargs)
    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    peft_params = dict(
        r=256,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj", "embed_tokens", "lm_head"],
        lora_alpha=32,
        lora_dropout=0.0,
        bias="none",
        use_gradient_checkpointing=False,
        random_state=42,
        use_rslora=True,
        loftq_config=None,
    )

    train_args = dict(
        per_device_eval_batch_size=1,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=1,
        num_train_epochs=1,
        warmup_steps=0,
        warmup_ratio=0.1,
        max_grad_norm=1.0,
        learning_rate=5e-5,
        optim = "adamw_torch_fused",
        weight_decay=0.0,
        lr_scheduler_type="cosine",
        seed=42,
        report_to="none",
        save_strategy="no",
        eval_strategy="no",
        logging_strategy="no",
        fp16=False,
        bf16=True,
        # Disable FSDP (use standard DDP)
        fsdp="",
        ddp_find_unused_parameters=False,
        dataloader_num_workers=0,
        gradient_checkpointing=False,
        half_precision_backend="cpu_amp",
        no_cuda=False,

    )

    max_seq_length = 8192

    # 모델 경로 fallback
    _model_paths = [
        "/kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
        "/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1",
    ]
    _model_path = None
    for _p in _model_paths:
        if os.path.exists(_p):
            _model_path = _p
            break
    if _model_path is None:
        raise FileNotFoundError(f"Model not found in: {_model_paths}")
    print(f"[Rank {rank}] model path: {_model_path}")

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=_model_path,
        full_finetuning=False,
        load_in_4bit=False,
        local_files_only=True,
        use_gradient_checkpointing=False,
        max_seq_length=max_seq_length,
        dtype=torch.bfloat16,
        attn_implementation="sdpa",

    )

    model = FastLanguageModel.get_peft_model(model, **peft_params)
#    model.config._attn_implementation = "eager"

    # for name, param in model.named_parameters():
    #     if param.dtype == torch.float32:
    #         param.data = param.data.to(torch.float16)

    default_weights = get_peft_model_state_dict(model, adapter_name="default")
    default_weights = {k: v.clone().detach() for k, v in default_weights.items()}

    collator = QwenDataCollatorForCompletionOnlyLM(
        tokenizer=tokenizer,
        mlm=False,
    )

    formatter = QwenFormatter(tokenizer=tokenizer)

    max_new_tokens = formatter.max_new_tokens()

    max_score = -np.log(0.2)

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    arc_test_set = ArcDataset.from_file(test_path)

    dir_outputs = "/kaggle/inference_outputs"
    os.makedirs(dir_outputs, exist_ok=True)

    while not queue.empty():

        if time.time() > end_time:
            print(f"[Rank {rank}] stop!")
            break

        key = queue.get()
        if key is None:
            break
        
        start_time = time.time()
        
        torch.cuda.reset_peak_memory_stats()

        load_result = set_peft_model_state_dict(
            model,
            default_weights.copy(),
            adapter_name="default",
        )

        model = FastLanguageModel.for_training(model)

        puzzle_ds = arc_test_set.change_keys([key])

        train_ds = puzzle_ds.augment(n=16, shfl_keys=True, seed=1)
        train_ds = train_ds.cut_to_len(formatter=formatter, name="text", max_len=max_seq_length)

        with io.StringIO() as buf, redirect_stdout(buf), redirect_stderr(buf):
            from accelerate.utils import GradScalerKwargs
            trainer_kwargs = {}
            trainer = UnslothFixedTrainer(
                model=model,
                tokenizer=tokenizer,
                data_collator=collator,
                train_dataset=Dataset.from_list(train_ds.as_list(formatter)),
                dataset_text_field="text",
                max_seq_length=max_seq_length,
                args=UnslothTrainingArguments(**train_args),
            )

            stats = trainer.train()

            model = trainer.accelerator.unwrap_model(model, keep_fp32_wrapper=False)

            del trainer

        model = FastLanguageModel.for_inference(model)
#        model.config._attn_implementation = "eager"
        
        gc.collect()
        torch.cuda.empty_cache()
            
        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for training")

        torch.cuda.reset_peak_memory_stats()
        
        print(f"[Rank {rank}] training stats for puzzle {key}: {stats}")

        puzzle_ds_multi = puzzle_ds.split_multi_replies()

        eval_ds = puzzle_ds_multi.augment(n=2, seed=2)
        eval_ds = eval_ds.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)

        test_id_to_subkeys = defaultdict(list)
        for subkey in sorted(eval_ds.keys):
            test_id = subkey.split(".")[0].split("_")[1]
            test_id_to_subkeys[test_id].append(subkey)

        batches = []
        for test_id, subkeys in test_id_to_subkeys.items():
            # 0: permute x 2
            # 4: rot90.rot90.permute x 2
            batch = []
            for offset in [0, 4]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 2: permute.rot90 x 2
            # 6: rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [2, 6]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
        for test_id, subkeys in test_id_to_subkeys.items():
            # 8: transpose.permute x 2
            # 12: transpose.rot90.rot90.permute x 2
            batch = []
            for offset in [8, 12]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)
            # 10: transpose.rot90.permute x 2
            # 14: transpose.rot90.rot90.rot90.permute x 2
            batch = []
            for offset in [10, 14]:
                batch.extend(subkeys[offset:offset+2])
            batches.append(batch)

        with torch.inference_mode():
                
            known_scores = {}

            for subkeys in batches:

                spend_time = time.time() - start_time
                if spend_time > 2500 or time.time() > end_time:
                    print(f"[Rank {rank}] timeout after {spend_time:.1f}s for puzzle {key}")
                    break

                print(f"[Rank {rank}] decoding {subkeys}")

                tokens = []
                for subkey in subkeys:
                    data = eval_ds.get(subkey, formatter)
                    tokens.append(tokenizer.encode(data["input"]))

                dfs_result = inference_turbo_dfs(model, tokens, max_new_tokens, max_score, end_time)

                for subkey_id, scored_beams in dfs_result:

                    subkey = subkeys[subkey_id]
                    bk = subkey.split(".")[0]
                    decoded_result = []

                    for beam_score, tokens in scored_beams:

                        array = formatter.convert_tokens_to_array(tokens)
                        if array is None:
                            continue

                        solution = puzzle_ds_multi.invert_mod(array, subkey, inv_perm=True)

                        grid_id = (bk, tuple(map(tuple, solution)))

                        if grid_id in known_scores:
                            augmented_scores = known_scores[grid_id]
                        else:
                            print(f"[Rank {rank}] scoring {subkey} #{len(decoded_result)}")
                            aug_dataset = ArcDataset(
                                keys=[bk],
                                queries={bk: puzzle_ds_multi.queries.get(bk)},
                                replies={bk: [solution.tolist()]},
                            )
                            aug_dataset = aug_dataset.augment(seed=hash(bk) % 1024**2)
                            aug_dataset = aug_dataset.cut_to_len(formatter=formatter, name="input", max_len=max_seq_length-max_new_tokens)
                            aug_queries = []
                            aug_answers = []
                            for augmented_sample in aug_dataset.as_list(formatter):
                                aug_queries.append(augmented_sample["input"])
                                aug_answers.append(augmented_sample["reply"])
                            augmented_scores1 = calc_scores(aug_queries[:4], aug_answers[:4], tokenizer, model)
                            augmented_scores2 = calc_scores(aug_queries[4:], aug_answers[4:], tokenizer, model)
                            augmented_scores = augmented_scores1 + augmented_scores2
                            known_scores[grid_id] = augmented_scores
                        
                        decoded_result.append({
                            "beam_score": beam_score,
                            "score_aug": augmented_scores,
                            "solution": solution,
                        })

                    if len(decoded_result):
                        with bz2.BZ2File(os.path.join(dir_outputs, subkey), "w") as f:
                            pickle.dump(decoded_result, f)

        memory_allocated = torch.cuda.max_memory_allocated() // 1024**2
        print(f"[Rank {rank}] allocated {memory_allocated}MB for inference")
        
        spend_time = time.time() - start_time
        print(f"[Rank {rank}] finished {key} in {spend_time:.1f}s")

Writing arc_solver.py


In [6]:
%%writefile starter.py
import os
import time
import json
import torch
import argparse
import torch.multiprocessing as mp


def local_worker(rank, queue, end_time):
    
    os.environ["CUDA_VISIBLE_DEVICES"] = str(rank)

    torch.set_default_device("cpu")

    # Fix Unsloth patching issue
    if rank > 0:
        while not os.path.exists(f"/kaggle/worker{rank-1}"):
            time.sleep(5)
    
    from arc_solver import worker

    with open(f"/kaggle/worker{rank}", "w") as f:
        f.write("Ok")
    
    print(f"[Rank {rank}] start!")
    
    worker(rank, queue, end_time)
    
    print(f"[Rank {rank}] done!")


if __name__ == "__main__":

    parser = argparse.ArgumentParser()
    parser.add_argument("--end-time", type=float, default=0.0)
    args = parser.parse_args()

    rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

    if rerun_mode:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json"
    else:
        test_path = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json"

    with open(test_path, "r") as f:
        data = json.load(f)

    queue = mp.Manager().Queue()

    for key in sorted(data.keys()):
        if not rerun_mode:
            if key not in ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]:
                continue
        queue.put(key)
    for _ in range(4):
        queue.put(None)
    
    mp.spawn(local_worker, args=(queue, args.end_time), nprocs=4)

Writing starter.py


In [7]:
# ============================================================
# [결합 스위프 셀 v2.4 - 색칠 토큰레벨 패치] kgmon 결합 노트북용. 위치:
#   v2.3: ① 색칠 페이즈 - U형 모양판 예측 top-2 를 shapecolor 어댑터에
#   중간 input 으로 넣어 원색 채색 (user턴 = input+빈줄+모양판, TTT 없음,
#   4뷰). Add Input 에 arc2-shapecolor 추가 시 자동 발동.
#   ② shapeuni 신판 우선(파일명 내림차순: shu4val > shu2val).
#   v2: 분류기 v2 내장 - kgmon 약점 계열(교차분석 실측: 이동1/14,
#   패턴복제1/15, 배경없음3/16, 가장큰도형0/6, output유사0/3, move4 0/2)을
#   해당 '계열 어댑터'(arc2-adapters-v2)로 집중 공략. 그 계열은 일반 항등
#   스위프에서 제외(중복 제거). Add Input 에 arc2-adapters-v2 추가 필요.
#   "%%writefile starter.py 셀 다음, !python starter.py 실행 셀 이전".
#   짝: 결합_병합_셀 v1 (맨 끝). Add Input: sft139 + arc2-shapeuni.
#
#   역할 (우리가 먼저, 싸게):
#     ① U형 쌍둥이 + 항등을 싼 설정(8뷰/TTT10/채점증강2)으로 스위프
#        -> /kaggle/inference_2step 에 후보 덤프 (kgmon 저장소와 무충돌)
#     ② 고신뢰 판정: 항등 8뷰 중 HICONF_K개 이상 동일 답 -> 그 태스크를
#        kgmon 큐 '맨 뒤'로 재배열 (빼지 않음 - 시간 남으면 kgmon 재검증)
#        -> kgmon 심층 시간이 어려운 태스크에 집중
#   예산 (kgmon 은 global_end_time 까지 나머지 전부 자체 관리):
#     실전 재실행: U형 2400s + 항등 3000s = 1.5h 고정
#     테스트 커밋: U형 700s + 항등 800s = 25분
#       (+ 오프라인 kgmon 은 TEST_KGMON_TASKS 4태스크만 -> 총 약 1h.
#        10h 테스트로 늘리려면: 아래 예산 상향 + TEST_KGMON_TASKS = None)
#   근거: 2step V4 27.92 회귀 사례 - 본대 시간을 뺏으면 역효과. 몫 고정 필수.
#   v3.1: NLL 채점 배치 분할(4개씩, OOM 시 반감) - 22GB 채점 OOM 10건 해소.
#   v2 -> v3 (정답수 증대):
#     ① 항등(원본) 페이즈 신규: 어댑터 병합 '없이' 순수 sft139 로 전 태스크의
#        원본 문제를 직접 예측 (TTT+16뷰+NLL 동일 적용). 지금까지는 변환판만
#        풀어서 발동 태스크 밖은 후보가 0이었음 - 커버리지 41유닛 -> 전 유닛.
#     ② 2페이즈 서브프로세스: 항등(어댑터 없음) 먼저 -> U형(병합) 나중.
#        프로세스 분리라 GPU 잔류 0 유지.
#     ③ NLL 을 토큰당 평균으로 정규화 - 제출 셀의 kgmon 식(득표-평균점수)
#        에서 득표가 정상 작동 (기존엔 총합 NLL 이 득표를 압도).
#     ④ (제출 셀 v2와 짝) 복원 후 프록시색 잔존 후보 패널티.
#   1등 solver 없이 우리 모델만 돌린다. 짝: 우리단독_제출_셀.py (셀2).
#   Add Input 2개 필요: sorokin/qwen3_4b_grids15_sft139 +
#   kwonyongdeuk/arc2-shapeuni. GPU 노트북.
#   v1 -> v2 (실측 3/41 대응): ① TTT-lite = 쌍둥이마다 chunk 쌍으로
#   소형 LoRA 즉석학습(1등의 태스크별 적응을 이식) ② 뷰 8->16
#   ③ 후보별 증강 NLL 실측(score_aug) -> 선택 손실 감소.
#   (global_end_time 이 1번 셀에서 고정되므로 시간 예산 수정 불필요.
#    병합 셀 v3.5 는 그대로 맨 끝.) GPU 필요.
#   v1.1 -> v1.2: 모델 로드/추론을 서브프로세스로 분리. v1.1 은 셀 안
#   에서 로드해 P/sd/enc/gen 참조가 GPU 0 에 7.1GB 잔류 -> 1등 solver
#   rank0 OOM 전멸 사고 (2026-07-30 로그). 프로세스 종료 = 전부 반환.
#   Add Input 필요: kwonyongdeuk/arc2-shapeuni (pyTorch/shu2val)
#   짝: 2단계 병합 셀 v3.5 (U형 mono 복원 지원. 2c/3c 는 v3.4 S경로 그대로)
#
#   목적: "output 만 collapse" 분해 - input 은 원본 그대로 두고 train output
#   만 모양판으로 바꾼 쌍둥이를 만들어, 통합 모양어댑터(shapeuni) 를 얹은
#   qwen 이 test 모양판을 예측한다. 역색칠(원색 복원)은 병합 셀이 한다.
#   어댑터가 1개라 스왑/라우팅 없음 - 문맥이 변환 종류를 지정한다.
#
#   변형 4종 (코랩 학습 v2 와 동일 정의, 고정 3분류 반영):
#     mono   : output 원색 유지, 고정①색 칸+고정②재등장 도형 -> 단색 m (통일판)
#     mono_a : 위와 같되 고정③ 앵커는 원색 유지 (앵커 보존판, 앵커 있을 때만)
#     3c     : 고정①색 -> m1 / 변동 -> m2
#     2c     : 비배경 전부 -> m (순수 실루엣)
#   등록: mono/mono_a = 종류 'U' (병합 v3.5 신규 복원),
#         2c/3c = 종류 'S' 스펙 (병합 v3.4 부터 있던 S 복원 경로 재사용 -
#         S 복원은 원본 input + 표만 쓰므로 input 원본 쌍둥이에도 그대로 성립)
#   게이트: train(chunk) 전쌍 왕복 복원 일치. 고정/재등장 판정은 모델이
#   문맥으로 보는 chunk 안에서만 도출 (학습과 동일 정보).
#
#   동작 순서:
#     1) U형 쌍둥이 도출 -> 분해_계획.json 등록 (키 {tid}u{j}zzz, 멱등)
#        * challenges_2step.json 에는 넣지 않음 - 1등 기본 실행이 풀 필요 없음
#     2) sft139 로드 + shapeuni LoRA 수동 병합 (peft 불필요)
#     3) 자체 추론: D8+색순열 8뷰 greedy -> 역증강 -> /kaggle/inference_2step
#        에 kgmon 후보 형식(bz2 pickle)으로 덤프 -> 병합 셀이 복원·합류
# ============================================================
MAX_VIRTUAL_U = 9999       # 상한 해제 - 발동하는 전 태스크
PER_TASK_MAX = 3           # 태스크당 쌍둥이 상한 (mono > mono_a > 3c > 2c 순)
N_VIEWS = 8                # [스위프] 뷰 수
TTT_STEPS = 10             # [스위프] 즉석학습 스텝 (0=끄기)
SCORE_AUGS = 2             # [스위프] 채점 증강
ORIG_ENABLE = True         # 항등(원본) 스위프 on/off
import os as _os0
_RERUN0 = bool(_os0.getenv("KAGGLE_IS_COMPETITION_RERUN"))
# ---- 스위프 예산 (초): 실전 = 1.5h 고정 / 테스트 = 25분 ----
U_BUDGET_S = 1200 if _RERUN0 else 600       # U형 (고유가치)
CAT_BUDGET_S = 3000 if _RERUN0 else 3600    # 계열 어댑터 페이즈 총예산
ORIG_BUDGET_S = 1200 if _RERUN0 else 600    # 항등 (타깃 계열 제외, 싼 것부터)
COLOR_BUDGET_S = 600 if _RERUN0 else 300    # 색칠(3페이즈): U형 모양판에 채색
COLOR_VIEWS = 4            # 색칠 뷰 수 / 모양판 후보 top-2
TARGET_CATS = {'이동': 'move1', '패턴복제': 'patterndup', '배경없음': 'nobg',
               'move3': 'move3'}
# 선정 원칙(사용자): 우리가 답을 낸 적 있는 계열만. 실측(eval):
#   이동 1/14, 패턴복제 1/15, 배경없음 2/16, move3 3/10(유일한 +gap)
#   제외: 가장큰도형 0/6, output유사 0/3, move4 0/2, 패턴삭제 0/2 (전부 0)
HICONF_K = 6               # 항등 8뷰 중 K개 이상 동일 답 = 고신뢰 -> 큐 뒤로
TEST_KGMON_TASKS = ["0934a4d8", "36a08778", "981571dc", "aa4ec2a5"]
#   오프라인 커밋에서 kgmon 이 풀 태스크 (None = 전체. 10h 테스트용)
print(f"스위프 모드: {'실전(1.5h)' if _RERUN0 else '테스트(25분)'} "
      f"- U형 {U_BUDGET_S}s / 항등 {ORIG_BUDGET_S}s")
LIMIT = 0                  # 0=전체(120태스크 발동분) / 1=가장 쉬운 쌍둥이 1개만
CTX_CELLS = 2200           # 문맥 train 쌍 셀 예산 (학습 chunk 2800 보다 보수적)
GEN_CELL_MAX = 1100        # 생성 토큰 상한

import os, re, json, glob, time, bz2, pickle, zlib
import numpy as np

# ---------------- 공통 유틸 (병합 셀과 동일 정의) ----------------
def _G(x):
    return np.asarray(x, dtype=int)

def _shapes(g, bg, conn=4):
    g = _G(g); H, W = g.shape
    NB = ((1, 0), (-1, 0), (0, 1), (0, -1)) if conn == 4 else \
         tuple((a, b) for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0))
    seen = np.zeros((H, W), bool); out = []
    for y in range(H):
        for x in range(W):
            if seen[y, x] or g[y, x] == bg:
                continue
            c0 = int(g[y, x]); st = [(y, x)]; seen[y, x] = True; cs = []
            while st:
                a, b = st.pop(); cs.append((a, b))
                for dy, dx in NB:
                    ny, nx = a + dy, b + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == c0:
                        seen[ny, nx] = True; st.append((ny, nx))
            out.append((c0, cs))
    return out

def _norm(cs):
    y0 = min(y for y, _ in cs); x0 = min(x for _, x in cs)
    return frozenset((y - y0, x - x0) for y, x in cs)

def _center(cs):
    return (sum(y for y, _ in cs) / len(cs), sum(x for _, x in cs) / len(cs))

def _S_silkey(cs):
    return json.dumps(sorted(map(list, _norm(cs))))

# ---------------- 분류기 v2 (내장, 네임스페이스 격리) ----------------
_CLS_SRC = r'''
# -*- coding: utf-8 -*-
"""코랩_어댑터_재학습_v2.py

Automatically generated by Colab.

Original file is located at
    https://colab.research.google.com/drive/1R6a7YedaYYPHCmJDmRZqH3wXymajDXGn
"""













# ============================================================================
# [분류기 v2] 사용자 사양(1~6번 순차, 조기 종료) 구현. 단일 라벨 출력.
#   순서: 복원 -> 그리드 -> (배경탐색) -> 색깔 -> 이동 -> move3 -> move2
#         -> move4 -> 도형/패턴 x 삭제/복제 -> output유사 -> 가장큰도형
#         -> 배경없음/분류불가
#   classify_v2(task) -> (라벨, 근거문자열)
# ============================================================================
import numpy as np
from collections import Counter
from itertools import combinations

class DiagonalShapeExtractor:
    """배경 제거 후 도형 추출 - 핵심 특징 추출기"""

    def __init__(self):
        self.background_threshold = 0.3
    def extract_tiny_diagonal_regions(self, matrix):
        """배경을 최소화한 tiny diagonal regions 추출"""

        if not matrix or not matrix[0]:
            return []

        from collections import Counter

        flat = [val for row in matrix for val in row]
        background_color = Counter(flat).most_common(1)[0][0]

        rows, cols = len(matrix), len(matrix[0])
        visited = [[False for _ in range(cols)] for _ in range(rows)]
        regions = []

        def is_valid(r, c):
            return 0 <= r < rows and 0 <= c < cols

        def dfs_diagonal(r, c, current_region_cells):
            if not is_valid(r, c) or visited[r][c] or matrix[r][c] == background_color:
                return

            visited[r][c] = True
            current_region_cells.append((r, c))

            for dr in [-1, 0, 1]:
                for dc in [-1, 0, 1]:
                    if dr == 0 and dc == 0:
                        continue
                    dfs_diagonal(r + dr, c + dc, current_region_cells)

        for r in range(rows):
            for c in range(cols):
                if matrix[r][c] != background_color and not visited[r][c]:
                    current_region_cells = []
                    dfs_diagonal(r, c, current_region_cells)

                    if current_region_cells:
                        min_r = min(cell[0] for cell in current_region_cells)
                        max_r = max(cell[0] for cell in current_region_cells)
                        min_c = min(cell[1] for cell in current_region_cells)
                        max_c = max(cell[1] for cell in current_region_cells)

                        region_h = max_r - min_r + 1
                        region_w = max_c - min_c + 1

                        region_matrix = [[background_color for _ in range(region_w)] for _ in range(region_h)]

                        for rr, cc in current_region_cells:
                            region_matrix[rr - min_r][cc - min_c] = matrix[rr][cc]

                        color = matrix[current_region_cells[0][0]][current_region_cells[0][1]]

                        regions.append({
                            'pixels': current_region_cells,
                            'size': (region_h, region_w),
                            'color': int(color),
                            'position': (min_r, min_c),
                            'grid': region_matrix
                        })

        return regions


    def extract_all_shapes(self, grid):
        """모든 diagonal 도형 추출"""

        if not grid or len(grid) == 0:
            return []

        import numpy as np

        arr = np.array(grid)
        h, w = arr.shape

        shapes = []

        unique_colors = np.unique(arr)

        for color in unique_colors:
            if color == 0:
                continue

            mask = (arr == color)

            coords = np.argwhere(mask)

            if len(coords) == 0:
                continue

            min_y, min_x = coords.min(axis=0)
            max_y, max_x = coords.max(axis=0)

            shape_h = int(max_y - min_y + 1)
            shape_w = int(max_x - min_x + 1)

            pixels = [(int(y), int(x)) for y, x in coords]

            shape_grid = []
            for i in range(shape_h):
                row = []
                for j in range(shape_w):
                    y = int(min_y) + i
                    x = int(min_x) + j
                    if (y, x) in pixels:
                        row.append(int(color))
                    else:
                        row.append(0)
                shape_grid.append(row)

            shapes.append({
                'pixels': pixels,
                'size': (shape_h, shape_w),
                'color': int(color),
                'position': (int(min_y), int(min_x)),
                'grid': shape_grid
            })

        return shapes

    # ✅ 이 메서드 추가
    def extract_all_regions(self, grid):
        """모든 Region 추출"""

        if not grid or len(grid) == 0:
            return []

        import numpy as np
        from scipy import ndimage

        arr = np.array(grid)
        h, w = arr.shape

        regions = []

        unique_colors = np.unique(arr)

        for color in unique_colors:
            if color == 0:
                continue

            mask = (arr == color)

            labeled, num_features = ndimage.label(mask)

            for region_id in range(1, num_features + 1):
                region_mask = (labeled == region_id)

                coords = np.argwhere(region_mask)
                pixels = [(int(y), int(x)) for y, x in coords]

                if len(pixels) == 0:
                    continue

                y_coords = [p[0] for p in pixels]
                x_coords = [p[1] for p in pixels]

                min_y, max_y = min(y_coords), max(y_coords)
                min_x, max_x = min(x_coords), max(x_coords)

                region_h = max_y - min_y + 1
                region_w = max_x - min_x + 1

                shape_grid = []
                for i in range(region_h):
                    row = []
                    for j in range(region_w):
                        y = min_y + i
                        x = min_x + j
                        if (y, x) in pixels:
                            row.append(int(color))
                        else:
                            row.append(0)
                    shape_grid.append(row)

                regions.append({
                    'pixels': pixels,
                    'size': (region_h, region_w),
                    'color': int(color),
                    'position': (min_y, min_x),
                    'grid': shape_grid
                })

        return regions



_DSE = DiagonalShapeExtractor()

def _to_ext_grid(g, bg):
    """추출기는 0을 배경으로 본다 -> 배경색을 0으로 리매핑 (0이 실색이면 10으로)."""
    g = np.asarray(g).copy()
    if bg is None:
        bg = -1
    if bg != 0:
        zero_mask = (g == 0)
        g[g == bg] = 0
        g[zero_mask] = 10
    return g.tolist()

def _regions_user(g, bg):
    """사용자 extract_all_regions (4연결, scipy) -> [(색, 정규화 frozenset)]"""
    try:
        regs = _DSE.extract_all_regions(_to_ext_grid(g, bg))
    except Exception:
        return None
    out = []
    for r in regs:
        px = r['pixels']
        y0 = min(y for y, _ in px); x0 = min(x for _, x in px)
        col = r['color']
        if col == 10:
            col = 0
        out.append((int(col), frozenset((y - y0, x - x0) for y, x in px)))
    return out

def find_mirror_axes(grid, unknown=(0,), max_off=7):
    """[사용자 절차] 중간선 오프셋 스캔 축 탐지.
    좌우축: 정확히 반 가르는 중간선부터 y좌표 +1..+max_off, -1..-max_off 이동하며
    미러링 일치 '개수'를 셈 -> 최다 위치가 축. 상하축도 동일. 미지색은 제외.
    반환: {'좌우축': {'위치'(반칸 단위 float), '오프셋', '일치수', '검사쌍', '일치율'},
           '상하축': {...}, '사분면': (r축, c축) 교차 정보}"""
    g = np.asarray(grid)
    H, W = g.shape
    unk = set(int(u) for u in unknown)
    known = np.ones(g.shape, bool)
    for u in unk:
        known &= (g != u)
    def scan(L, axis):
        center_p2 = L - 1                      # 정확히 반 가르는 중간선 (2p 좌표)
        best = None
        order = [0] + [s * k for k in range(1, max_off + 1) for s in (1, -1)]
        for off in order:
            p2 = center_p2 + off
            if not (1 <= p2 <= 2 * L - 3): continue
            cnt = tot = 0
            for a in range(L):
                b = p2 - a
                if not (0 <= b < L) or a >= b: continue
                A = g[a] if axis == 'r' else g[:, a]
                B = g[b] if axis == 'r' else g[:, b]
                kA = known[a] if axis == 'r' else known[:, a]
                kB = known[b] if axis == 'r' else known[:, b]
                m = kA & kB
                tot += int(m.sum())
                cnt += int((A[m] == B[m]).sum())
            if best is None or cnt > best['일치수']:
                best = {'위치': p2 / 2.0, '오프셋': off, '일치수': cnt,
                        '검사쌍': tot, '일치율': round(cnt / tot, 3) if tot else 0.0}
        return best
    r_ax = scan(H, 'r')   # 상하 가르는 축 (행)
    c_ax = scan(W, 'c')   # 좌우 가르는 축 (열)
    return {'상하축': r_ax, '좌우축': c_ax,
            '사분면_중심': (r_ax['위치'] if r_ax else None,
                        c_ax['위치'] if c_ax else None)}


def check_restore_quadrant(train, accept=0.80, reject=0.70):
    """[사용자 사양] 복원 실행 없이 판정:
    가림색 후보(출력 근거: 소멸색/bbox일치) -> 축 스캔(find_mirror_axes)
    -> 사분면 양축 일치율로 판정.
      - 양축 모두 accept(80%) 이상 & 검사쌍 충분 -> 복원 (True)
      - 최선축이 reject(70%) 미만 -> 아님 (False)
      - 후보 없음/표본 부족/중간값 -> 포기 (False)
    모든 train 쌍이 통과해야 True."""
    if not train:
        return False
    for e in train:
        i = np.asarray(e['input']); o = np.asarray(e['output'])
        # 가림색 후보 (출력 근거)
        cands = []
        for c in np.unique(i).tolist():
            n_i = int((i == c).sum())
            if n_i == 0: continue
            if i.shape == o.shape:
                if int((o == c).sum()) == 0:
                    cands.append(int(c))                 # 소멸색
            else:
                ys, xs = np.where(i == c)
                if (int(ys.max()-ys.min()+1), int(xs.max()-xs.min()+1)) == o.shape:
                    cands.append(int(c))                 # bbox == 출력 크기
        if not cands:
            return False                                  # 포기
        ok_pair = False
        need = i.size // 8                                # 최소 검사쌍
        for occ in cands:
            ax = find_mirror_axes(i, unknown=(occ,))
            ra, ca = ax['상하축'], ax['좌우축']
            if not ra or not ca:
                continue                                  # 축 못 구함 -> 포기
            if ra['검사쌍'] < need or ca['검사쌍'] < need:
                continue                                  # 표본 부족 -> 포기
            lo = min(ra['일치율'], ca['일치율'])
            if lo < reject:
                continue                                  # 70% 미만 -> 아님
            if lo >= accept:
                ok_pair = True; break                     # 양축 80%+ -> 복원
            # 70~80% 사이 = 중간값 -> 이 후보로는 판정 보류
        if not ok_pair:
            return False
    return True

def _objs8(g, bg=0):
    g = np.asarray(g); H, W = g.shape
    seen = np.zeros((H, W), bool); out = []
    for r in range(H):
        for c in range(W):
            if g[r, c] != bg and not seen[r, c]:
                st = [(r, c)]; seen[r, c] = True; cells = []
                while st:
                    y, x = st.pop(); cells.append((y, x))
                    for dy in (-1, 0, 1):
                        for dx in (-1, 0, 1):
                            ny, nx = y + dy, x + dx
                            if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                               and g[ny, nx] != bg:
                                seen[ny, nx] = True; st.append((ny, nx))
                r0 = min(y for y, _ in cells); c0 = min(x for _, x in cells)
                sil = frozenset((y - r0, x - c0) for y, x in cells)
                colored = frozenset((y - r0, x - c0, int(g[y, x])) for y, x in cells)
                pos = frozenset(cells)
                out.append((sil, colored, pos))
    return out



# ---------------- 공통 도구 ----------------

def _grid_lines(g):
    """꽉 찬 단색 가로/세로 구분선 (행/열 인덱스, 색)"""
    g = np.asarray(g)
    rows = [(r, int(g[r, 0])) for r in range(g.shape[0])
            if len(set(g[r].tolist())) == 1]
    cols = [(c, int(g[0, c])) for c in range(g.shape[1])
            if len(set(g[:, c].tolist())) == 1]
    return rows, cols

def _grid_lines_info(g):
    """내부 단색 꽉 찬 선 -> {색: (가로선그룹위치, 세로선그룹위치)}"""
    g = np.asarray(g)
    H, W = g.shape
    rows, cols = _grid_lines(g)
    rows = [(r, c) for r, c in rows if 0 < r < H - 1]
    cols = [(x, c) for x, c in cols if 0 < x < W - 1]
    by = {}
    for r, c in rows:
        by.setdefault(c, [set(), set()])[0].add(r)
    for x, c in cols:
        by.setdefault(c, [set(), set()])[1].add(x)
    def groups(s):
        s = sorted(s)
        out = []
        for v in s:
            if out and v - out[-1][-1] == 1:
                out[-1].append(v)
            else:
                out.append([v])
        return [gr[0] for gr in out]
    info = {}
    for c, (rs, cs) in by.items():
        if (g == c).mean() >= 0.5:
            continue
        info[c] = (groups(rs), groups(cs))
    return info

def _grid_task(ins):
    _ins_ref = ins
    """2번 그리드 (재정의): 가로+세로 교차 -> 그리드.
    한 방향뿐이면 전 train 에서 위치 비율(1/2, 1/3 지점) 또는
    절대 위치가 모두 같아야 그리드."""
    per = []
    for g in ins:
        info = _grid_lines_info(np.asarray(g))
        best = None
        for c, (rs, cs) in info.items():
            if rs and cs:
                best = ('both', None)
                break
            if rs:
                H = np.asarray(g).shape[0]
                best = best or ('row', tuple(rs),
                                tuple(round((r + 0.5) / H, 2) for r in rs))
            elif cs:
                W = np.asarray(g).shape[1]
                best = best or ('col', tuple(cs),
                                tuple(round((x + 0.5) / W, 2) for x in cs))
        if best is None:
            return False
        per.append(best)
    if all(p[0] == 'both' for p in per):
        return True
    if any(p[0] == 'both' for p in per):
        return False
    dirs = {p[0] for p in per}
    if len(dirs) != 1:
        return False
    abspos = {p[1] for p in per}
    ratios = {p[2] for p in per}
    if len(abspos) == 1 or len(ratios) == 1:
        return True
    # 위치가 달라도: 선색이 '정확히 관통선만' 이루면(울퉁불퉁 없음) 인정
    def _clean_through(g):
        g = np.asarray(g)
        H, W = g.shape
        for c in set(g.ravel().tolist()):
            mask = (g == c)
            if not mask.any() or mask.mean() >= 0.5:
                continue
            full_rows = [r for r in range(H) if mask[r].all()]
            full_cols = [x for x in range(W) if mask[:, x].all()]
            line_set_r = set(full_rows); line_set_c = set(full_cols)
            for r in full_rows:
                if not (0 < r < H - 1):
                    continue
                bump = False
                for dr in (-1, 1):
                    rr = r + dr
                    if rr in line_set_r or not (0 <= rr < H):
                        continue
                    if (g[rr] == c).any():
                        bump = True
                if not bump:
                    return True
            for x in full_cols:
                if not (0 < x < W - 1):
                    continue
                bump = False
                for dx in (-1, 1):
                    xx = x + dx
                    if xx in line_set_c or not (0 <= xx < W):
                        continue
                    if (g[:, xx] == c).any():
                        bump = True
                if not bump:
                    return True
        return False
    return all(_clean_through(g) for g in _ins_ref)

def _find_bg(i, o=None):
    """3번: 배경 탐색 6단계. (색, 단계번호) 또는 None.
    각 단계: 최다색부터, 전체의 20% 이상 + 모서리 영역(1/4/9칸) 4곳 중
    2곳 이상에서 지배색이면 배경."""
    i = np.asarray(i)
    total = i.size
    cnt = Counter(i.ravel().tolist())

    def corner_hits(c, k):
        regs = [i[:k, :k], i[:k, -k:], i[-k:, :k], i[-k:, -k:]]
        return sum(1 for r in regs if r.size and (r == c).mean() > 0.5)

    def scan(allowed, base):
        for k_i, k in enumerate((1, 2, 3)):
            for c, n in cnt.most_common():
                if c not in allowed or n < 0.2 * total:
                    continue
                if corner_hits(c, k) >= 2:
                    return int(c), base + k_i + 1
        return None

    all_colors = set(cnt)
    r = scan(all_colors, 0)
    if r:
        return r
    # 고정색 제외 재시도 (output==input 인 색 제외, 같은 크기일 때만)
    if o is not None:
        o = np.asarray(o)
        if o.shape == i.shape:
            fixed = {c for c in all_colors
                     if bool(np.array_equal(i == c, o == c))}
            r = scan(all_colors - fixed, 3)
            if r:
                return r
    return None

def _shapes(g, bg):
    """도형 = 4방향 연결, 3칸 이상 (사용자 정의: 대각선 제외).
    반환: [(색, 정규화좌표 frozenset)]"""
    r = _regions_user(g, bg)
    if r is not None:
        return [(c, s) for c, s in r if len(s) >= 3]
    g = np.asarray(g)
    H, W = g.shape
    seen = np.zeros((H, W), bool)
    out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg:
                continue
            col = g[r, c]
            stack, cells = [(r, c)], []
            seen[r, c] = True
            while stack:
                y, x = stack.pop()
                cells.append((y, x))
                for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == col:
                        seen[ny, nx] = True
                        stack.append((ny, nx))
            if len(cells) >= 3:
                y0 = min(y for y, _ in cells)
                x0 = min(x for _, x in cells)
                out.append((int(col),
                            frozenset((y - y0, x - x0) for y, x in cells)))
    return out


def _shapes_all(g, bg):
    """4연결 성분 전부 (크기 제한 없음). [(색, 정규화 frozenset)]"""
    r = _regions_user(g, bg)
    if r is not None:
        return r
    g = np.asarray(g)
    H, W = g.shape
    seen = np.zeros((H, W), bool)
    out = []
    for r in range(H):
        for c in range(W):
            if seen[r, c] or g[r, c] == bg:
                continue
            col = g[r, c]
            stack, cells = [(r, c)], []
            seen[r, c] = True
            while stack:
                y, x = stack.pop()
                cells.append((y, x))
                for dy, dx in ((1, 0), (-1, 0), (0, 1), (0, -1)):
                    ny, nx = y + dy, x + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == col:
                        seen[ny, nx] = True
                        stack.append((ny, nx))
            y0 = min(y for y, _ in cells)
            x0 = min(x for _, x in cells)
            out.append((int(col),
                        frozenset((y - y0, x - x0) for y, x in cells)))
    return out

def _d4(sh):
    """실루엣 D4 정규형"""
    pts = np.array(sorted(sh))
    best = None
    for k in range(4):
        for fl in (False, True):
            p = pts.copy()
            for _ in range(k):
                p = np.stack([p[:, 1], -p[:, 0]], 1)
            if fl:
                p = np.stack([p[:, 0], -p[:, 1]], 1)
            p = p - p.min(0)
            key = tuple(sorted(map(tuple, p.tolist())))
            if best is None or key < best:
                best = key
    return best

def _resize_nn(g, shape):
    g = np.asarray(g)
    ys = (np.arange(shape[0]) * g.shape[0] / shape[0]).astype(int)
    xs = (np.arange(shape[1]) * g.shape[1] / shape[1]).astype(int)
    return g[np.ix_(ys, xs)]

def _outsim(outs, th=0.95):
    """6번 (재정의): output끼리 - 축소/확대/그대로 + 회전/미러 후 모양 동일.
    색은 달라도 됨(함수적 색매핑). 전부 일직선이면 인정."""
    if len(outs) < 2:
        return False
    if all(min(o.shape) == 1 for o in outs):
        return True                      # 전부 일직선
    def pair_best(a, b):
        best = 0.0
        for k in range(4):
            for fl in (False, True):
                bb = np.rot90(b, k)
                if fl:
                    bb = bb[:, ::-1]
                if bb.shape != a.shape:
                    bb = _resize_nn(bb, a.shape)
                m, ok = {}, 0
                for va, vb in zip(a.ravel().tolist(), bb.ravel().tolist()):
                    if va in m:
                        if m[va] == vb:
                            ok += 1
                    else:
                        m[va] = vb
                        ok += 1
                best = max(best, ok / a.size)
        return best
    if any(len(np.unique(o)) < 2 for o in outs):
        return False                     # 단색 output은 무의미
    return min(pair_best(a, b) for a, b in combinations(outs, 2)) >= th

def _largest_rule(ins, outs, bgs):
    """6번 가장큰도형 (재정의+확장):
    게이트: output이 input보다 작아야 함(같으면 불인정), 전 쌍.
    (a) 쌍별 최대 도형(연결 단위 또는 색깔 단위 union)의 bbox ~= output (7%)
    (b) 어떤 '한 색'이 전 쌍에서 union bbox ~= output (가장 크지 않아도 됨,
        같은 색이 배경 위에 여러 개면 전체 선택)"""
    def _gate(i, o):
        return (o.shape[0] <= i.shape[0] and o.shape[1] <= i.shape[1]
                and o.shape != i.shape)

    def bbox_of_color(i, c):
        ys, xs = np.where(i == c)
        if len(ys) == 0:
            return None
        return (int(ys.max() - ys.min() + 1), int(xs.max() - xs.min() + 1))

    def close(osh, bb, gate_ok=True):
        # k=1: 게이트(출력<입력) 필요, 7% 허용
        if gate_ok and \
           abs(osh[0] - bb[0]) <= max(1, 0.07 * bb[0]) and \
           abs(osh[1] - bb[1]) <= max(1, 0.07 * bb[1]):
            return True
        # k=2,3 정수배 확대: 게이트 불필요, +-1칸 정밀 일치
        for k in (2, 3):
            if abs(osh[0] - bb[0] * k) <= 1 and abs(osh[1] - bb[1] * k) <= 1:
                return True
        return False

    # (a) 쌍별 최대 도형
    ok_a = True
    for i, o, bg in zip(ins, outs, bgs):
        cands = []
        shp = _shapes(i, bg if bg is not None else -1)
        if shp:
            big = max(shp, key=lambda s: len(s[1]))
            cands.append((max(y for y, _ in big[1]) + 1,
                          max(x for _, x in big[1]) + 1))
        best_c, best_n = None, 0
        for c in np.unique(i).tolist():
            if bg is not None and c == bg:
                continue
            n = int((i == c).sum())
            if n > best_n:
                best_n, best_c = n, c
        if best_c is not None:
            cands.append(bbox_of_color(i, best_c))
        gk = _gate(i, o)
        if not any(b and close(o.shape, b, gk) for b in cands):
            ok_a = False
            break
    if ok_a:
        return True

    # (b) 일관된 한 색의 union bbox == output (전 쌍)
    colors = set()
    for i in ins:
        colors |= set(np.unique(i).tolist())
    for c in colors:
        good = True
        for i, o, bg in zip(ins, outs, bgs):
            if bg is not None and c == bg:
                good = False
                break
            bb = bbox_of_color(i, c)
            if bb is None or not close(o.shape, bb, _gate(i, o)):
                good = False
                break
        if good:
            return True
    return False

# ---------------- 본체 ----------------

def classify_v2(task, explain=False):
    train = task.get('train', [])
    if not train:
        return ('분류불가', '빈 train') if explain else '분류불가'
    ins = [np.asarray(e['input']) for e in train]
    outs = [np.asarray(e['output']) for e in train]

    def ret(label, why):
        return (label, why) if explain else label

    # ---- 1번 복원 (재정의): 축 스캔 -> 사분면 미러 대응 만장일치(가림색 제외) ----
    def _restore_unanimous_v2(i, o):
        i = np.asarray(i); o = np.asarray(o)
        if i.size <= 81:                 # 9x9 이하는 복원 아님
            return False
        # 가림색 후보 = 출력에서 소멸하는 색 (없으면 0).
        # (대칭/사분면 검사 자체는 input에만 적용 - output 대칭은 요구하지 않음)
        occs = [c for c in np.unique(i).tolist()
                if c not in np.unique(o).tolist()]
        if not occs:
            occs = [0] if 0 in np.unique(i).tolist() else []
        for occ in occs:
            ax = find_mirror_axes(i, unknown=(occ,))
            ra, ca = ax.get('상하축'), ax.get('좌우축')
            if not ra or not ca:
                continue
            ry, cx = ra['위치'], ca['위치']
            H, W = i.shape
            groups_ok = groups_tot = 0
            seen = set()
            bad = False
            for y in range(H):
                for x in range(W):
                    my = int(round(2 * ry - y)); mx = int(round(2 * cx - x))
                    pts = {(y, x)}
                    if 0 <= my < H: pts.add((my, x))
                    if 0 <= mx < W: pts.add((y, mx))
                    if 0 <= my < H and 0 <= mx < W: pts.add((my, mx))
                    key = frozenset(pts)
                    if key in seen or len(pts) < 2:
                        continue
                    seen.add(key)
                    vals = {int(i[p]) for p in pts if int(i[p]) != occ}
                    if len(vals) >= 1:
                        groups_tot += 1
                        if len(vals) == 1:
                            groups_ok += 1
                        else:
                            bad = True
            if groups_tot >= 8 and groups_ok / groups_tot >= 0.98:
                return True
        return False
    try:
        if all(_restore_unanimous_v2(e['input'], e['output']) for e in train):
            return ret('복원', '1번: 사분면 만장일치(가림색 제외)')
    except Exception:
        pass

    # ---- 2번 그리드 ----
    if _grid_task(ins):
        return ret('그리드', '2번: 전 입력에 구분선 그리드')

    # ---- 3번 배경 ----
    bgs = [(_find_bg(i, o) or (None, 0)) for i, o in zip(ins, outs)]
    bg_cols = [b[0] for b in bgs]
    has_bg = all(c is not None for c in bg_cols)
    same = all(i.shape == o.shape for i, o in zip(ins, outs))

    if has_bg and same:
        bmask_i = [i == c for i, c in zip(ins, bg_cols)]
        bmask_o = [o == c for o, c in zip(outs, bg_cols)]
        # ---- 4번 색깔: 배경 위치와 양이 고정 ----
        if all(np.array_equal(a, b) for a, b in zip(bmask_i, bmask_o)):
            return ret('색깔', '4번: 배경 마스크 동일')
        # ---- 4번 이동: 배경 양만 같고 위치 다름 ----
        if all(a.sum() == b.sum() for a, b in zip(bmask_i, bmask_o)):
            return ret('이동', '4번: 배경 양 동일·위치 상이')

    # ---- 5번 move2/move3 (동일색 지우기): input==output 셀을 지우고
    #      남은 색(배경 제외)이 1색 -> move2 / 2색+총량보존 -> move3 ----
    if has_bg and same:
        verdicts = []
        for i, o, bg in zip(ins, outs, bg_cols):
            d = (i != o)
            if not d.any():
                verdicts.append(None); continue
            rem = (set(int(v) for v in o[d].ravel()) |
                   set(int(v) for v in i[d].ravel()))
            rem.discard(int(bg))
            if len(rem) == 1:
                verdicts.append('move2')
            elif len(rem) == 2 and all(
                    int((i == c).sum()) == int((o == c).sum()) for c in rem):
                verdicts.append('move3')
            else:
                verdicts.append(None)
        if verdicts and all(v is not None for v in verdicts):
            if 'move3' in verdicts:
                return ret('move3', '5번: 동일색 지우기-2색 총량보존')
            return ret('move2', '5번: 동일색 지우기-잔여 1색')

    if has_bg:
        sh_i = [_shapes(i, c) for i, c in zip(ins, bg_cols)]
        sh_o = [_shapes(o, c) for o, c in zip(outs, bg_cols)]

        def multiset_in(sub, sup):          # sub의 모든 원소가 sup에 (중복 포함)
            pool = list(sup)
            for x in sub:
                if x in pool:
                    pool.remove(x)
                else:
                    return False
            return True

        # ---- 5번 move3 (재정의): output 모든 도형이 input에 존재(회전·미러
        #      허용) + 그 도형들을 지우면 배경색만 남음 ----
        def _move3_pair(i, o, bg):
            oc = _shapes_all(o, bg)          # 크기 제한 없는 성분
            if not oc or all(len(cells) < 3 for c, cells in oc):
                return False
            in_d4 = [(c, _d4(cells)) for c, cells in _shapes_all(i, bg)]
            pool = list(in_d4)
            for c, cells in oc:              # 전 성분(점 포함)이 input에 존재
                key = (c, _d4(cells))
                if key in pool:
                    pool.remove(key)
                else:
                    return False
            return True
        if all(_move3_pair(i, o, c) for i, o, c in zip(ins, outs, bg_cols)):
            return ret('move3', '5번: 전 도형 보존(D4)+잔여없음')
        # ---- 5번 move2: 비고정 색 정확히 2개 + 이동색 총량 보존 ----
        if same:
            ok2 = True
            for i, o, bg in zip(ins, outs, bg_cols):
                cols = set(np.unique(i).tolist()) | set(np.unique(o).tolist())
                nonfix = [c for c in cols
                          if not np.array_equal(i == c, o == c)]
                if len(cols) < 3 or len(nonfix) != 2:
                    ok2 = False
                    break
                mv = [c for c in nonfix if c != bg]
                if len(mv) == 1 and \
                   int((i == mv[0]).sum()) != int((o == mv[0]).sum()):
                    ok2 = False
                    break
            if ok2:
                return ret('move2', '5번: 비고정 2색+총량 보존')
        # ---- 5번 move4: 실루엣은 전부 보존, 색은 다름 ----
        d4_i = [[_d4(s[1]) for s in si] for si in sh_i]
        d4_o = [[_d4(s[1]) for s in so] for so in sh_o]
        if all(so for so in sh_o) and \
           all(multiset_in(do, di) for di, do in zip(d4_i, d4_o)) and \
           any(not multiset_in(so, si) for si, so in zip(sh_i, sh_o)):
            return ret('move4', '5번: 실루엣 보존·색 변경')

        # ---- 5번 배경 양 증감 -> 삭제/복제, 도형/패턴 구분 ----
        if same:
            bi = [(i == c).sum() for i, c in zip(ins, bg_cols)]
            bo = [(o == c).sum() for o, c in zip(outs, bg_cols)]
            grew = all(b > a for a, b in zip(bi, bo))
            shrank = all(b < a for a, b in zip(bi, bo))
            if grew or shrank:
                # input==output 부분을 지우고, 달라진 부분만 남겨 성분 분석.
                # 도형 = 3칸+ 4연결이며 직선/대각선/점이 아님.
                # 패턴요소 = 점(1~2칸)·직선(1xN)·대각선.
                votes = []
                for i, o, c in zip(ins, outs, bg_cols):
                    diff = (i != o)
                    srcg = np.where(diff, i if grew else o, c)  # 변화분만
                    comps = _shapes(srcg, c)          # 3칸+ 4연결 성분
                    shape_cells = 0
                    for col, cells in comps:
                        ys = [y for y, _ in cells]
                        xs = [x for _, x in cells]
                        h = max(ys) + 1
                        w = max(xs) + 1
                        line = (h == 1 or w == 1)
                        diag = (h == w and len(cells) == h and
                                (all(y == x for y, x in cells) or
                                 all(y == w - 1 - x for y, x in cells)))
                        if not line and not diag:
                            shape_cells += len(cells)
                    votes.append(shape_cells >= 3)     # 도형 성분 존재 여부
                is_shape = sum(votes) * 2 > len(votes)   # 쌍 다수결
                if grew:
                    return ret('도형삭제' if is_shape else '패턴삭제',
                               '5번: 배경 증가')
                return ret('도형복제' if is_shape else '패턴복제',
                           '5번: 배경 감소')

    # ---- 6번 output유사 ----
    if _outsim(outs):
        return ret('output유사', '6번: output끼리 유사')
    # ---- 6번 가장큰도형 ----
    try:
        if _largest_rule(ins, outs, bg_cols):
            return ret('가장큰도형', '6번: 최대도형 재등장/bbox')
    except Exception:
        pass
    # ---- 6번 폴백 ----
    if not has_bg:
        return ret('배경없음', '6번: 배경 탐색 실패')
    return ret('분류불가', '6번: 잔여')



# -*- coding: utf-8 -*-

'''
_CLS_NS = {}
exec(_CLS_SRC, _CLS_NS)
classify_v2 = _CLS_NS['classify_v2']

# ---------------- 변환 정의 (코랩 학습 v2 와 동일 - 0 = 배경) ----------------
class DiagonalShapeExtractor:
    def extract_all_shapes(self, grid):
        if not grid or len(grid) == 0:
            return []
        arr = np.array(grid); shapes = []
        for color in np.unique(arr):
            if color == 0:
                continue
            coords = np.argwhere(arr == color)
            if len(coords) == 0:
                continue
            shapes.append({'pixels': [(int(y), int(x)) for y, x in coords],
                           'color': int(color)})
        return shapes

    def extract_all_regions(self, grid):
        if not grid or len(grid) == 0:
            return []
        from scipy import ndimage
        arr = np.array(grid); regions = []
        for color in np.unique(arr):
            if color == 0:
                continue
            labeled, n = ndimage.label(arr == color)
            for rid in range(1, n + 1):
                px = [(int(y), int(x)) for y, x in np.argwhere(labeled == rid)]
                if px:
                    regions.append({'pixels': px, 'color': int(color)})
        return regions

_DSE = DiagonalShapeExtractor()

def _unit_key(u):
    px = u['pixels']
    y0 = min(y for y, _ in px); x0 = min(x for _, x in px)
    return (u['color'], frozenset((y - y0, x - x0) for y, x in px))

def _cols_of(chunk):
    used = set()
    for p in chunk:
        used |= set(np.unique(_G(p['input'])).tolist())
        used |= set(np.unique(_G(p['output'])).tolist())
    return {int(c) for c in used}

def _free(chunk, n):
    used = _cols_of(chunk)
    fr = [c for c in range(9, 0, -1) if c not in used]
    return fr[:n] if len(fr) >= n else None

def _fixed_cols(chunk):
    F = []
    for c in sorted(_cols_of(chunk) - {0}):
        ok, present = True, False
        for p in chunk:
            i, o = _G(p['input']), _G(p['output'])
            if i.shape != o.shape or not np.array_equal(i == c, o == c):
                ok = False; break
            if (i == c).any():
                present = True
        if ok and present:
            F.append(c)
    return F

def _partial_anchor_cells(i, o):
    if i.shape != o.shape:
        return set()
    anchors = set()
    try:
        out_pos = {(u['color'], frozenset(u['pixels']))
                   for u in _DSE.extract_all_regions(o.tolist())}
        for u in _DSE.extract_all_regions(i.tolist()):
            c = u['color']; key = (c, frozenset(u['pixels']))
            if key not in out_pos:
                continue
            if np.array_equal(i == c, o == c):
                continue
            anchors.add(key)
    except Exception:
        pass
    return anchors

def _mono_mask(p, F, anchor_keep):
    i, o = _G(p['input']), _G(p['output'])
    mono = np.zeros(o.shape, bool)
    if F and i.shape == o.shape:
        mono |= np.isin(o, F)
    try:
        in_keys = {_unit_key(u) for u in _DSE.extract_all_regions(i.tolist())} | \
                  {_unit_key(u) for u in _DSE.extract_all_shapes(i.tolist())}
        o_list = o.tolist()
        for u in (_DSE.extract_all_regions(o_list)
                  + _DSE.extract_all_shapes(o_list)):
            if _unit_key(u) in in_keys:
                for y, x in u['pixels']:
                    mono[y, x] = True
    except Exception:
        pass
    if anchor_keep:
        for c, px in _partial_anchor_cells(i, o):
            for y, x in px:
                mono[y, x] = False
    return mono

def _tf_mono(chunk, keep_anchor):
    fr = _free(chunk, 1)
    if not fr:
        return None
    m = fr[0]; F = _fixed_cols(chunk)
    out, changed, has_anchor = [], False, False
    for p in chunk:
        i, o = _G(p['input']), _G(p['output'])
        if keep_anchor and _partial_anchor_cells(i, o):
            has_anchor = True
        mono = _mono_mask(p, F, anchor_keep=keep_anchor)
        if mono.any():
            changed = True
        o2 = o.copy(); o2[mono] = m
        out.append({'input': p['input'], 'output': o2.tolist()})
    if keep_anchor and not has_anchor:
        return None
    return (out, m, F) if changed else None

def _tf_2c(chunk):
    if len(_cols_of(chunk) - {0}) < 2:
        return None
    fr = _free(chunk, 1)
    if not fr:
        return None
    m = fr[0]
    out = [{'input': p['input'],
            'output': np.where(_G(p['output']) != 0, m, 0).tolist()}
           for p in chunk]
    return out, m

def _tf_3c(chunk):
    F = _fixed_cols(chunk)
    rest = sorted(_cols_of(chunk) - {0} - set(F))
    if not F or not rest:
        return None
    fr = _free(chunk, 2)
    if not fr:
        return None
    m1, m2 = fr[0], fr[1]
    out = []
    for p in chunk:
        o = _G(p['output']).copy()
        fmask = np.isin(o, F); vmask = (o != 0) & ~fmask
        o[fmask] = m1; o[vmask] = m2
        out.append({'input': p['input'], 'output': o.tolist()})
    return out, m1, m2, F, rest

# ---------------- 복원 (게이트/오프라인 측정용 - 병합 셀과 동일 로직) --------
def _S_sil_paint(p, i, bg, m, colors):
    if not colors:
        return
    pool = {}
    for c, cs in _shapes(i, bg):
        if c in colors:
            pool.setdefault(_norm(cs), []).append((c, cs))
    for cs in [cs for c, cs in _shapes(p, bg) if c == m]:
        cand = pool.get(_norm(cs), [])
        if not cand:
            continue
        if len({c for c, _ in cand}) == 1:
            used = cand[0]
        else:
            cy, cx = _center(cs)
            used = min(cand, key=lambda t: (_center(t[1])[0] - cy) ** 2
                                           + (_center(t[1])[1] - cx) ** 2)
        cand.remove(used)
        for y, x in cs:
            p[y, x] = used[0]

def _S_groups(spec):
    v = spec['변형']; F, M, V = spec['F'], spec['M'], spec['V']
    if v == '2색':
        return [], F + M + V
    if v == '3색':
        return F, M + V
    if v == '3색도형':
        return F + M, V
    return [], M + V

def _restore_S(pred, spec, orig_in):
    p = _G(pred).copy(); i = _G(orig_in); bg = spec['bg']
    G1, G2 = _S_groups(spec)
    m1, m2 = spec.get('m1'), spec['m2']
    same = (p.shape == i.shape)
    if m1 is not None and same:
        mask = (p == m1) & np.isin(i, spec['F'])
        p[mask] = i[mask]
    if m1 is not None:
        _S_sil_paint(p, i, bg, m1, [c for c in spec['M'] if c in G1])
    mode = spec['변동복원']
    if mode != '모양색':
        _S_sil_paint(p, i, bg, m2, G2)
    if mode == '모양색':
        f = spec['모양표']
        for cs in [cs for c, cs in _shapes(p, bg) if c == m2]:
            v = f.get(_S_silkey(cs))
            if v is not None:
                for y, x in cs:
                    p[y, x] = v
    if mode in ('위치', '위치단색') and same:
        mask = (p == m2) & np.isin(i, G2)
        p[mask] = i[mask]
    if mode == '전이' and same:
        f = spec['전이표']
        ys, xs = np.where(p == m2)
        for y, x in zip(ys.tolist(), xs.tolist()):
            v = f.get(str(int(i[y, x])))
            if v is not None:
                p[y, x] = v
    if mode in ('단색', '위치단색') and spec.get('단색') is not None:
        p[p == m2] = spec['단색']
    return p

def _restore_U(pred, spec, orig_in):
    """U형(mono 계열): m 칸 = ①input F색 자리 되칠 ②잔여 m 성분은 input
    도형(4연결, bg=0) 실루엣 매칭 (동형 다색 -> 위치 최근접)"""
    p = _G(pred).copy(); i = _G(orig_in); m = spec['m']
    if p.shape == i.shape and spec['F']:
        mask = (p == m) & np.isin(i, spec['F'])
        p[mask] = i[mask]
    _S_sil_paint(p, i, 0, m, sorted({int(c) for c in np.unique(i) if c != 0}))
    return p

def restore_any(pred, spec, orig_in):
    if spec['종류'] == 'I':
        return _G(pred)
    if spec['종류'] == 'U':
        return _restore_U(pred, spec, orig_in)
    return _restore_S(pred, spec, orig_in)

# ---------------- 스펙 도출 (chunk 기준) + 왕복 게이트 ----------------
def _var_restore_tables(chunk, spec):
    """m2 변동복원 모드 도출 (S셀과 동일: 단색/전이/모양색/위치/위치단색).
    통과 스펙 반환 또는 None"""
    same_all = all(_G(p['input']).shape == _G(p['output']).shape
                   for p in chunk)
    def apply_out(o):
        o = _G(o).copy()
        G1, G2 = _S_groups(spec)
        if G1:
            o[np.isin(o, G1)] = spec['m1']
        if G2:
            o[np.isin(o, G2)] = spec['m2']
        return o
    solid, trans, silmap = set(), ({} if same_all else None), {}
    for p in chunk:
        i, o = _G(p['input']), _G(p['output'])
        to = apply_out(o)
        m2m = (to == spec['m2'])
        solid |= {int(v) for v in o[m2m]}
        if trans is not None:
            for a, b in zip(i[m2m].tolist(), o[m2m].tolist()):
                k = str(int(a))
                if trans.get(k, int(b)) != int(b):
                    trans = None; break
                trans[k] = int(b)
        if silmap is not None:
            for c, cs in _shapes(to, spec['bg']):
                if c != spec['m2']:
                    continue
                oc = {int(o[y, x]) for y, x in cs}
                if len(oc) != 1:
                    silmap = None; break
                k = _S_silkey(cs); v = oc.pop()
                if silmap.get(k, v) != v:
                    silmap = None; break
                silmap[k] = v
    c_star = int(next(iter(solid))) if len(solid) == 1 else None
    modes = (['단색'] if c_star is not None else []) + \
            (['전이'] if trans else []) + \
            (['모양색'] if silmap else []) + ['위치'] + \
            (['위치단색'] if c_star is not None else [])
    for mode in modes:
        sp = dict(spec); sp['변동복원'] = mode; sp['단색'] = c_star
        if mode == '전이':
            sp['전이표'] = trans
        if mode == '모양색':
            sp['모양표'] = silmap
        ok = True
        for p in chunk:
            try:
                to = apply_out(p['output'])
                if not np.array_equal(_restore_S(to, sp, p['input']),
                                      _G(p['output'])):
                    ok = False; break
            except Exception:
                ok = False; break
        if ok:
            return sp
    return None

def derive_U_twins(chunk):
    """[(spec, 변환된 chunk, 꼬리표)] - 우선순위 mono_a > 3c > mono.
    변형별 실측(19.83 완주): mono_a 3/13, 3c 1/3, mono 2/21, 2c 0/2(제거)"""
    out = []
    r = _tf_mono(chunk, keep_anchor=True)                  # mono_a (앵커 보존)
    if r:
        tc, m, F = r
        sp = {'종류': 'U', '변형': 'mono_a', 'F': F, 'm': m, 'bg': 0}
        if all(np.array_equal(_restore_U(t['output'], sp, p['input']),
                              _G(p['output'])) for t, p in zip(tc, chunk)):
            out.append((sp, tc, 'Umono_a'))
    r = _tf_3c(chunk)                                      # 3c -> S'3색' 스펙
    if r:
        tc, m1, m2, F, V = r
        sp0 = {'종류': 'S', '변형': '3색', 'F': F, 'M': [], 'V': V,
               'bg': 0, 'm1': m1, 'm2': m2}
        sp = _var_restore_tables(chunk, sp0)
        if sp:
            out.append((sp, tc, f"U3c/{sp['변동복원']}"))
    r = _tf_mono(chunk, keep_anchor=False)                 # mono (통일판)
    if r:
        tc, m, F = r
        sp = {'종류': 'U', '변형': 'mono', 'F': F, 'm': m, 'bg': 0}
        if all(np.array_equal(_restore_U(t['output'], sp, p['input']),
                              _G(p['output'])) for t, p in zip(tc, chunk)):
            out.append((sp, tc, 'Umono'))
    # 동일 변환판 중복 제거 (mono == mono_a 가 되는 경우 등)
    seen, ded = set(), []
    for sp, tc, tag in out:
        key = json.dumps([t['output'] for t in tc], sort_keys=True)
        if key in seen:
            continue
        seen.add(key); ded.append((sp, tc, tag))
    return ded

def pick_chunk(train):
    """문맥 = 학습 chunk 와 동일 조건 (<=4쌍, 셀 예산). 결정적 선택."""
    sel, cells = [], 0
    for p in train:
        if len(sel) >= 4:
            break
        c = (len(p['input']) * len(p['input'][0]) +
             len(p['output']) * len(p['output'][0]))
        if c > CTX_CELLS or cells + c > CTX_CELLS:
            continue
        sel.append(p); cells += c
    return sel if len(sel) >= 2 else None

# ---------------- 1) 쌍둥이 도출 + 분해_계획 등록 (멱등) ----------------
_rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
_BASE = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/"
_SRC = _BASE + ("arc-agi_test_challenges.json" if _rerun
                else "arc-agi_evaluation_challenges.json")
_CH = json.load(open(_SRC))
try:
    _plan = json.load(open("/kaggle/working/분해_계획.json"))
except Exception:
    _plan = {}

for _tid in list(_plan):                      # 기존 U 쌍둥이 제거 (멱등)
    _plan[_tid] = [tw for tw in _plan[_tid]
                   if not re.match(r".*[uocx]\d+zzz$", tw["k"])]
    if not _plan[_tid]:
        _plan.pop(_tid)

TWINS = []                                    # (key, tid, spec, chunk_t, tag)
for _tid in sorted(_CH):
    if sum(1 for t in TWINS if t[1] == _tid) >= PER_TASK_MAX:
        continue
    train = _CH[_tid].get('train', [])
    chunk = pick_chunk(train)
    if chunk is None:
        continue
    try:
        cands = derive_U_twins(chunk)
    except Exception:
        continue
    _j = 0
    for sp, tc, tag in cands[:PER_TASK_MAX]:
        if len(TWINS) >= MAX_VIRTUAL_U:
            break
        key = f"{_tid}u{_j}zzz"; _j += 1
        TWINS.append((key, _tid, sp, tc, tag))
        _plan.setdefault(_tid, []).append({"k": key, "spec": sp, "tag": tag})
    if len(TWINS) >= MAX_VIRTUAL_U:
        break

OTW = []                                      # 항등(원본) - 전 태스크
if ORIG_ENABLE:
    for _tid in sorted(_CH):
        train = _CH[_tid].get('train', [])
        chunk = pick_chunk(train)
        if chunk is None:                     # 폴백: 작은 쌍부터 최대 3000셀
            prs = sorted(train, key=lambda p:
                         len(p['input']) * len(p['input'][0]) +
                         len(p['output']) * len(p['output'][0]))
            sel, cells = [], 0
            for p in prs:
                c = (len(p['input']) * len(p['input'][0]) +
                     len(p['output']) * len(p['output'][0]))
                if cells + c > 3000 or len(sel) >= 4:
                    break
                sel.append(p); cells += c
            chunk = sel or None
        if chunk is None:
            continue
        key = f"{_tid}o0zzz"
        sp = {'종류': 'I'}
        OTW.append((key, _tid, sp, chunk, 'orig'))
        _plan.setdefault(_tid, []).append({"k": key, "spec": sp, "tag": 'orig'})

CTW = []                                      # 계열 타깃 (어댑터 라우팅)
CAT_OF = {}
for _tid in sorted(_CH):
    try:
        _lab = classify_v2({'train': _CH[_tid].get('train', [])[:4]})
    except Exception:
        _lab = '분류불가'
    if _lab in TARGET_CATS:
        CAT_OF[_tid] = _lab
for key, tid, sp, tc, tag in list(OTW):       # 타깃 태스크: c 쌍둥이로 이관
    if tid in CAT_OF:
        ckey = f"{tid}c0zzz"
        CTW.append((ckey, tid, {'종류': 'I'}, tc, f"cat:{CAT_OF[tid]}"))
        _plan.setdefault(tid, []).append(
            {"k": ckey, "spec": {'종류': 'I'}, "tag": f"cat:{CAT_OF[tid]}"})
OTW = [t for t in OTW if t[1] not in CAT_OF]  # 일반 항등에서 제외 (중복 제거)
json.dump(CAT_OF, open('/kaggle/working/타깃계열.json', 'w'), ensure_ascii=False)
from collections import Counter as _C0
print('계열 타깃:', dict(_C0(CAT_OF.values())), f'/ 총 {len(CAT_OF)}태스크')

with open("/kaggle/working/분해_계획.json", "w") as _f:
    json.dump(_plan, _f)

def _twin_cost(t):
    """쉬운(작은) 것부터: 문맥 셀 수 + test 셀 수*2 오름차순"""
    _, tid, _, tc, _ = t
    c = sum(len(x['input']) * len(x['input'][0]) +
            len(x['output']) * len(x['output'][0]) for x in tc)
    c += max(len(te['input']) * len(te['input'][0])
             for te in _CH[tid]['test']) * 2
    return c
TWINS.sort(key=_twin_cost)                    # 예산 안에서 최대한 많이
OTW.sort(key=_twin_cost)
from collections import Counter
print(f"U형 쌍둥이 {len(TWINS)}개 ({len({t[1] for t in TWINS})}태스크): "
      f"{dict(Counter(t[4].split('/')[0] for t in TWINS))} "
      f"/ 항등 {len(OTW)}태스크")

# ---------------- 2) 추론 서브프로세스 (GPU 잔류 0 보장) ----------------
_INFER_SRC = r'''# shape_infer.py v2 - TTT-lite + 16뷰 + 후보 NLL 채점
import os, json, glob, time, bz2, pickle, zlib
import numpy as np
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PL = json.load(open('/kaggle/working/shape_payload.json'))
N_VIEWS = PL['views']; BUDGET_S = PL['budget']
TTT_STEPS = PL.get('ttt_steps', 0); SCORE_AUGS = PL.get('score_augs', 4)

_cfgs = sorted({os.path.dirname(p) for p in glob.glob(
    '/kaggle/input/**/config.json', recursive=True)})
_cfgs.sort(key=lambda d: 'sft139' not in d)
MODEL_DIR = _cfgs[0] if _cfgs else None
assert MODEL_DIR and 'sft139' in MODEL_DIR, f'sft139 없음: {_cfgs[:3]}'
tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
tok.padding_side = 'left'
if tok.pad_token_id is None:
    tok.pad_token = tok.eos_token
mdl = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, torch_dtype=torch.bfloat16, local_files_only=True).cuda().eval()

# tied embed/lm_head 분리 (LoRA 델타 겹침 방지)
if PL.get('adapter') and getattr(mdl.config, 'tie_word_embeddings', False):
    import torch.nn as nn
    W0 = mdl.get_input_embeddings().weight
    nh = nn.Linear(W0.shape[1], W0.shape[0], bias=False,
                   dtype=W0.dtype, device=W0.device)
    nh.weight.data = W0.data.clone()
    mdl.set_output_embeddings(nh)
    mdl.config.tie_word_embeddings = False
    print('tied lm_head 분리 완료')

# shapeuni 전역 어댑터를 가중치에 병합 (베이스로 흡수)
if PL.get('adapter'):
    sd = torch.load(PL['adapter'], map_location='cpu')
    SCALE = 32.0 / 256.0
    P = dict(mdl.named_parameters())
    n_mg = 0
    with torch.no_grad():
        for k in list(sd):
            if 'lora_A' not in k and 'lora_embedding_A' not in k:
                continue
            k2 = k.replace('.default', '')
            emb = 'lora_embedding_A' in k2
            kb = k2.replace('lora_embedding_A', 'lora_embedding_B') if emb \
                 else k2.replace('lora_A', 'lora_B')
            base = k2.split('.lora_')[0].replace('base_model.model.', '')
            W = P.get(base + '.weight')
            assert W is not None, '병합 대상 없음: ' + base
            d = (sd[kb].float() @ sd[k].float())
            if emb:
                d = d.T
            assert d.shape == W.shape, f'{base}: {d.shape} vs {W.shape}'
            W += (SCALE * d).to(W.dtype).to(W.device)
            n_mg += 1
    print('LoRA 병합 %d층 <- %s' % (n_mg, os.path.basename(PL['adapter'])))
else:
    print('항등 페이즈: 어댑터 병합 없음 (순수 sft139)')
IM_END = tok.convert_tokens_to_ids('<|im_end|>')
ASST = tok.encode('<|im_start|>assistant\n', add_special_tokens=False)
ENDT = tok.encode('<|im_end|>', add_special_tokens=False)[0]

# TTT-lite 준비: 소형 LoRA (쌍둥이마다 리셋)
HAVE_TTT = TTT_STEPS > 0
if HAVE_TTT:
    try:
        from peft import (LoraConfig, get_peft_model,
                          get_peft_model_state_dict, set_peft_model_state_dict)
        try:
            mdl.enable_input_require_grads()
        except Exception:
            pass
        mdl.gradient_checkpointing_enable()
        lcfg = LoraConfig(r=32, lora_alpha=32, lora_dropout=0.0, bias='none',
                          target_modules=['q_proj', 'k_proj', 'v_proj',
                                          'o_proj', 'gate_proj', 'up_proj',
                                          'down_proj'])
        mdl = get_peft_model(mdl, lcfg)
        INIT = {k: v.clone() for k, v in get_peft_model_state_dict(mdl).items()}
        print('TTT-lite 준비: LoRA r=32, %d스텝/쌍둥이' % TTT_STEPS)
    except Exception as e:
        print('peft 사용 불가 -> TTT 생략:', str(e)[:120])
        HAVE_TTT = False

def _gs(g):
    return '\n'.join(''.join(str(int(v)) for v in row) for row in g)

def _aug(g, k, fl, perm):
    a = np.rot90(np.asarray(g, dtype=int), k)
    if fl:
        a = np.fliplr(a)
    return perm[a]

def _deaug(g, k, fl, perm):
    ip = np.zeros(10, int); ip[perm] = np.arange(10)
    a = ip[np.asarray(g, dtype=int)]
    if fl:
        a = np.fliplr(a)
    return np.rot90(a, -k)

def _parse(txt):
    txt = txt.split('<|im_end|>')[0].strip()
    rows = [r.strip() for r in txt.split('\n') if r.strip()]
    if not rows or any(not r.isdigit() for r in rows):
        return None
    w = len(rows[0])
    if any(len(r) != w for r in rows) or len(rows) > 30 or w > 30:
        return None
    return [[int(ch) for ch in r] for r in rows]

def _ctx_text(chunk, k, fl, perm, order=None):
    t = ''
    for oi in (order if order is not None else range(len(chunk))):
        x = chunk[oi]
        t += ('<|im_start|>user\n' + _gs(_aug(x['input'], k, fl, perm))
              + '<|im_end|><|im_start|>assistant\n'
              + _gs(_aug(x['output'], k, fl, perm)) + '<|im_end|>')
    return t

def _labels(ids):
    lab = [-100] * len(ids); j = 0
    while j <= len(ids) - len(ASST):
        if ids[j:j + len(ASST)] == ASST:
            s = j + len(ASST); e = s
            while e < len(ids) and ids[e] != ENDT:
                e += 1
            for t in range(s, min(e + 1, len(ids))):
                lab[t] = ids[t]
            j = e
        j += 1
    return lab

def _pad(seqs, labs=None):
    ml = max(len(x) for x in seqs); pad = tok.pad_token_id or 0
    ii = torch.full((len(seqs), ml), pad, dtype=torch.long)
    am = torch.zeros((len(seqs), ml), dtype=torch.long)
    lb = torch.full((len(seqs), ml), -100, dtype=torch.long)
    for i, s in enumerate(seqs):
        ii[i, :len(s)] = torch.tensor(s); am[i, :len(s)] = 1
        if labs is not None:
            lb[i, :len(labs[i])] = torch.tensor(labs[i])
    return ii.cuda(), am.cuda(), lb.cuda()

def _ttt(chunk, rng):
    """쌍둥이별 즉석학습: chunk 쌍을 D8+색순열+순서셔플 증강해 소형 LoRA 학습"""
    set_peft_model_state_dict(mdl, {k: v.clone() for k, v in INIT.items()})
    texts = []
    for _ in range(24):
        k, fl = int(rng.randint(4)), bool(rng.randint(2))
        perm = np.concatenate([[0], rng.permutation(np.arange(1, 10))])
        texts.append(_ctx_text(chunk, k, fl, perm,
                               order=rng.permutation(len(chunk))))
    seqs = [tok.encode(t)[:3600] for t in texts]
    labs = [_labels(s) for s in seqs]
    mdl.train()
    opt = torch.optim.AdamW([p for p in mdl.parameters() if p.requires_grad],
                            lr=1e-4)
    bi = 0
    for _ in range(TTT_STEPS):
        idx = [(bi + z) % len(seqs) for z in range(2)]; bi += 2
        ii, am, lb = _pad([seqs[z] for z in idx], [labs[z] for z in idx])
        out = mdl(input_ids=ii, attention_mask=am, labels=lb)
        out.loss.backward(); opt.step(); opt.zero_grad(set_to_none=True)
    mdl.eval()
    del opt
    torch.cuda.empty_cache()

@torch.no_grad()
def _nll(chunk, test_in, grids, rng):
    """각 후보의 증강 문맥 '토큰당' NLL (v3.1: 배치 분할 + OOM 적응.
    구판은 후보 전부 한 배치 -> 22GB GPU 에서 채점 OOM 10건 발생)"""
    res = [[] for _ in grids]
    for a in range(SCORE_AUGS):
        k, fl = a % 4, a >= 4
        perm = np.concatenate([[0], rng.permutation(np.arange(1, 10))])
        q = (_ctx_text(chunk, k, fl, perm) + '<|im_start|>user\n'
             + _gs(_aug(test_in, k, fl, perm))
             + '<|im_end|><|im_start|>assistant\n')
        qt = tok.encode(q)
        seqs, spans = [], []
        for g in grids:
            at = tok.encode(_gs(_aug(g, k, fl, perm)) + '<|im_end|>')
            seqs.append(qt + at); spans.append((len(qt), len(at)))
        gi, bs = 0, 4
        while gi < len(seqs):
            j2 = min(gi + bs, len(seqs))
            try:
                ii, am, _ = _pad(seqs[gi:j2])
                logits = mdl(input_ids=ii, attention_mask=am).logits.float()
                logp = torch.log_softmax(logits, -1).cpu()
                del logits
                for z in range(j2 - gi):
                    s, n = spans[gi + z]
                    tgt = torch.tensor(seqs[gi + z][s:s + n])
                    lp = logp[z, s - 1:s - 1 + n]
                    res[gi + z].append(
                        float(-lp[torch.arange(n), tgt].sum()) / max(n, 1))
                gi = j2
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache()
                if bs > 1:
                    bs = max(1, bs // 2)       # 배치 반감 후 재시도
                else:
                    res[gi].append(1.0)        # 1개도 안 되면 중립값
                    gi += 1
        torch.cuda.empty_cache()
    return res

OUTD = '/kaggle/inference_2step'
os.makedirs(OUTD, exist_ok=True)
t0 = time.time(); n_dump = n_unit = 0
for tw in PL['twins']:
    if tw.get('mode') == 'color':               # 색칠: (input+모양판)->원색
        if time.time() - t0 > BUDGET_S:
            print('시간 예산 소진 -', tw['key'], '부터 생략'); break
        rng = np.random.RandomState(zlib.crc32(tw['key'].encode()) % (2 ** 31))
        _nl = tok.encode('0\n0', add_special_tokens=False)
        NL_ID = _nl[1] if len(_nl) >= 3 else tok.convert_tokens_to_ids('\u010a')
        # 빈 줄('\n\n')은 문자열로 넣으면 초소형 토크나이저(WordLevel)가
        # UNK 오류를 냄 -> 개행 토큰 2개를 토큰 레벨로 직접 삽입 (v2.4 패치)
        for ti_idx, te in enumerate(tw['tests']):
            seqs, views = [], []
            for shp in te['shapes']:
                for v in range(PL.get('color_views', 4)):
                    k, fl = v % 4, (v // 4) % 2 == 1
                    perm = np.concatenate([[0],
                                           rng.permutation(np.arange(1, 10))])
                    ids = []
                    for p in tw['ctx3']:
                        ids += tok.encode('<|im_start|>user\n'
                                          + _gs(_aug(p['input'], k, fl, perm)))
                        ids += [NL_ID, NL_ID]
                        ids += tok.encode(_gs(_aug(p['shape'], k, fl, perm)))
                        ids += tok.encode('<|im_end|><|im_start|>assistant\n'
                                          + _gs(_aug(p['output'], k, fl, perm))
                                          + '<|im_end|>')
                    ids += tok.encode('<|im_start|>user\n'
                                      + _gs(_aug(te['input'], k, fl, perm)))
                    ids += [NL_ID, NL_ID]
                    ids += tok.encode(_gs(_aug(shp, k, fl, perm)))
                    ids += tok.encode('<|im_end|><|im_start|>assistant\n')
                    seqs.append(ids); views.append((k, fl, perm))
            sols = []
            try:
                for b0 in range(0, len(seqs), 8):
                    sub = seqs[b0:b0 + 8]
                    ml = max(len(s) for s in sub)
                    pad = tok.pad_token_id or 0
                    ii = torch.full((len(sub), ml), pad, dtype=torch.long)
                    am = torch.zeros((len(sub), ml), dtype=torch.long)
                    for zi, s in enumerate(sub):        # 좌측 패딩 (생성용)
                        ii[zi, ml - len(s):] = torch.tensor(s)
                        am[zi, ml - len(s):] = 1
                    with torch.no_grad():
                        gen = mdl.generate(input_ids=ii.cuda(),
                                           attention_mask=am.cuda(),
                                           do_sample=False,
                                           max_new_tokens=tw['cap'],
                                           eos_token_id=IM_END,
                                           pad_token_id=tok.pad_token_id,
                                           use_cache=True)
                    new = gen[:, ml:]
                    for vi, row in enumerate(new):
                        g = _parse(tok.decode(row, skip_special_tokens=False))
                        if g is None:
                            continue
                        k, fl, perm = views[b0 + vi]
                        try:
                            sols.append(_deaug(g, k, fl, perm).tolist())
                        except Exception:
                            continue
            except Exception as e:
                print('  %s_%d 색칠 실패: %s' % (tw['key'], ti_idx, str(e)[:80]))
                torch.cuda.empty_cache(); continue
            if not sols:
                continue
            outs = [{'solution': g, 'score_aug': [0.5]} for g in sols]
            with bz2.BZ2File('%s/%s_%d.shape' % (OUTD, tw['key'], ti_idx),
                             'w') as f:
                pickle.dump(outs, f)
            n_dump += len(outs); n_unit += 1
        continue
    if time.time() - t0 > BUDGET_S:
        print('시간 예산 소진 -', tw['key'], '부터 생략'); break
    rng = np.random.RandomState(zlib.crc32(tw['key'].encode()) % (2 ** 31))
    if HAVE_TTT:
        try:
            ts = time.time(); _ttt(tw['chunk'], rng)
            print('  TTT %s %ds' % (tw['key'], int(time.time() - ts)))
        except Exception as e:
            print('  TTT 실패(생성은 진행) %s: %s' % (tw['key'], str(e)[:80]))
            torch.cuda.empty_cache()
    for ti_idx, test_in in enumerate(tw['tests']):
        prompts, views = [], []
        for v in range(N_VIEWS):
            k, fl = v % 4, (v // 4) % 2 == 1
            perm = np.concatenate([[0], rng.permutation(np.arange(1, 10))])
            t = (_ctx_text(tw['chunk'], k, fl, perm)
                 + '<|im_start|>user\n' + _gs(_aug(test_in, k, fl, perm))
                 + '<|im_end|><|im_start|>assistant\n')
            prompts.append(t); views.append((k, fl, perm))
        sols = []                                  # 뷰별 원방향 모양판
        try:
            for b0 in range(0, N_VIEWS, 8):        # 8개씩 나눠 생성
                enc = tok(prompts[b0:b0 + 8], return_tensors='pt',
                          padding=True)
                enc = {a: b.cuda() for a, b in enc.items()}
                with torch.no_grad():
                    gen = mdl.generate(**enc, do_sample=False,
                                       max_new_tokens=tw['cap'],
                                       eos_token_id=IM_END,
                                       pad_token_id=tok.pad_token_id,
                                       use_cache=True)
                new = gen[:, enc['input_ids'].shape[1]:]
                for vi, row in enumerate(new):
                    g = _parse(tok.decode(row, skip_special_tokens=False))
                    if g is None:
                        continue
                    k, fl, perm = views[b0 + vi]
                    try:
                        sols.append(_deaug(g, k, fl, perm).tolist())
                    except Exception:
                        continue
        except Exception as e:
            print('  %s_%d 추론 실패: %s' % (tw['key'], ti_idx, str(e)[:80]))
            torch.cuda.empty_cache(); continue
        if not sols:
            continue
        uniq = []
        for g in sols:                             # 고유 후보 -> NLL 채점
            if g not in uniq:
                uniq.append(g)
        try:
            sc = _nll(tw['chunk'], test_in, uniq, rng)
        except Exception as e:
            print('  %s_%d 채점 실패(1.0 대체): %s'
                  % (tw['key'], ti_idx, str(e)[:80]))
            torch.cuda.empty_cache()
            sc = [[1.0]] * len(uniq)
        outs = [{'solution': g, 'score_aug': sc[uniq.index(g)]} for g in sols]
        with bz2.BZ2File('%s/%s_%d.shape' % (OUTD, tw['key'], ti_idx),
                         'w') as f:
            pickle.dump(outs, f)
        n_dump += len(outs); n_unit += 1
print('추론 완료: %d유닛 / 후보 %d개 덤프 (%ds, TTT=%s)'
      % (n_unit, n_dump, int(time.time() - t0), HAVE_TTT))
'''

if LIMIT:
    TWINS = TWINS[:LIMIT]
    OTW = OTW[:LIMIT]
ADPT = sorted(glob.glob("/kaggle/input/**/shapeuni_*.pt", recursive=True),
              key=lambda p: os.path.basename(p), reverse=True)  # 신판 우선
if not ADPT:
    print("*** shapeuni_*.pt 없음: Add Input -> Models -> "
          "kwonyongdeuk/arc2-shapeuni 추가하라. U형 페이즈 생략 ***")

with open('/kaggle/working/shape_infer.py', 'w') as _f:
    _f.write(_INFER_SRC)
import subprocess, sys

def _run_phase(twins, budget, adapter, label):
    """페이즈별 서브프로세스 (프로세스 종료 = GPU 전부 반환)"""
    if not twins or budget <= 0:
        print(f'[{label}] 대상 없음/예산 0 - 생략'); return
    payload = {'budget': budget, 'views': N_VIEWS, 'adapter': adapter,
               'ttt_steps': TTT_STEPS, 'score_augs': SCORE_AUGS, 'twins': []}
    for key, tid, sp, tc, tag in twins:
        mx = max(len(t['output']) * (len(t['output'][0]) + 1) for t in tc)
        payload['twins'].append(
            {'key': key, 'tag': tag,
             'cap': min(GEN_CELL_MAX, int(mx * 1.5) + 32),
             'chunk': tc,
             'tests': [te['input'] for te in _CH[tid]['test']]})
    with open('/kaggle/working/shape_payload.json', 'w') as _f:
        json.dump(payload, _f)
    _env = dict(os.environ, TOKENIZERS_PARALLELISM='false')
    _t = time.time()
    _rc = subprocess.run([sys.executable, 'shape_infer.py'],
                         cwd='/kaggle/working', env=_env).returncode
    print(f'[{label}] 페이즈 종료 rc={_rc} ({int(time.time() - _t)}s)')

if ADPT:
    _run_phase(TWINS, U_BUDGET_S, ADPT[0], 'U형')  # ① 고유가치 먼저

# ①-b 색칠 페이즈: U형 모양판 예측 top-2 에 shapecolor 로 채색
CADPT = sorted(glob.glob("/kaggle/input/**/shapecolor_*.pt", recursive=True))
if CADPT and ADPT:
    import bz2 as _bzc, pickle as _pkc
    from collections import Counter as _Ccc
    _ctwins, _xj = [], {}
    for key, tid, sp, tc, tag in TWINS:
        chunk = pick_chunk(_CH[tid].get('train', []))
        if chunk is None or len(chunk) != len(tc):
            continue
        tests = []
        for ti_idx, te in enumerate(_CH[tid]['test']):
            fp = f'/kaggle/inference_2step/{key}_{ti_idx}.shape'
            if not os.path.exists(fp):
                tests = []; break
            try:
                with _bzc.BZ2File(fp) as f:
                    outs = _pkc.load(f)
            except Exception:
                tests = []; break
            cnt = _Ccc(json.dumps(o['solution']) for o in outs)
            top = [json.loads(s) for s, _ in cnt.most_common(2)]
            tests.append({'input': te['input'], 'shapes': top})
        if not tests:
            continue
        _j = _xj.get(tid, 0); _xj[tid] = _j + 1
        xkey = f"{tid}x{_j}zzz"
        mx = max(len(p['output']) * (len(p['output'][0]) + 1) for p in chunk)
        _ctwins.append({'key': xkey, 'tag': f'C{tag}', 'mode': 'color',
                        'cap': min(GEN_CELL_MAX, int(mx * 1.5) + 32),
                        'ctx3': [{'input': o['input'], 'shape': t['output'],
                                  'output': o['output']}
                                 for o, t in zip(chunk, tc)],
                        'tests': tests})
        _plan.setdefault(tid, []).append(
            {"k": xkey, "spec": {'종류': 'U색'}, "tag": f'C{tag}'})
    with open("/kaggle/working/분해_계획.json", "w") as _f:
        json.dump(_plan, _f)
    if _ctwins:
        payload = {'budget': COLOR_BUDGET_S, 'views': N_VIEWS,
                   'adapter': CADPT[-1], 'ttt_steps': 0,
                   'score_augs': SCORE_AUGS, 'color_views': COLOR_VIEWS,
                   'twins': _ctwins}
        with open('/kaggle/working/shape_payload.json', 'w') as _f:
            json.dump(payload, _f)
        _env = dict(os.environ, TOKENIZERS_PARALLELISM='false')
        _t = time.time()
        _rc = subprocess.run([sys.executable, 'shape_infer.py'],
                             cwd='/kaggle/working', env=_env).returncode
        print(f'[색칠] 페이즈 종료 rc={_rc} ({int(time.time() - _t)}s, '
              f'{len(_ctwins)}쌍둥이)')
elif ADPT:
    print('shapecolor_*.pt 없음 - 색칠 페이즈 생략 '
          '(Add Input: arc2-shapecolor)')
# ② 계열 어댑터 페이즈들 (계열별 서브프로세스, 예산은 태스크수 비례)
_cats = {}
for t in CTW:
    _cats.setdefault(t[4].split(':')[1], []).append(t)
_n_ct = sum(len(v) for v in _cats.values()) or 1
for _cat, _tw in sorted(_cats.items(), key=lambda x: -len(x[1])):
    _slug = TARGET_CATS[_cat]
    _ap = sorted(glob.glob(f"/kaggle/input/**/{_slug}_*.pt", recursive=True))
    if not _ap:
        print(f'[계열:{_cat}] {_slug}_*.pt 없음 - 항등으로 환원')
        OTW += _tw
        continue
    _bud = max(150, int(CAT_BUDGET_S * len(_tw) / _n_ct))
    _tw.sort(key=_twin_cost)
    _run_phase(_tw, _bud, _ap[0], f'계열:{_cat}')
_run_phase(OTW, ORIG_BUDGET_S, None, '항등')     # ③ 잔여 태스크 항등 스위프
RUN = bool(TWINS or OTW)

# ---------------- 고신뢰 판정 + kgmon 큐 재배열/오프라인 대상 패치 ----------
import bz2 as _bz2m, pickle as _pkm, re as _rem
from collections import Counter as _Cc
HC = []
for key, tid, sp, tc, tag in (OTW + CTW):       # 항등/계열 뷰 수렴도 기준
    ok_all, got = True, False
    for ti_idx in range(len(_CH[tid]['test'])):
        fp = f'/kaggle/inference_2step/{key}_{ti_idx}.shape'
        if not os.path.exists(fp):
            ok_all = False; break
        try:
            with _bz2m.BZ2File(fp) as f:
                outs = _pkm.load(f)
        except Exception:
            ok_all = False; break
        got = True
        cnt = _Cc(json.dumps(o['solution']) for o in outs)
        if not cnt or max(cnt.values()) < HICONF_K:
            ok_all = False; break
    if got and ok_all:
        HC.append(tid)
json.dump(HC, open('/kaggle/working/highconf.json', 'w'))
print(f'고신뢰({HICONF_K}/{N_VIEWS}뷰 일치) {len(HC)}태스크 -> kgmon 큐 뒤로')

_s = open('starter.py').read()
_tgt = "    for key in sorted(data.keys()):"
if _tgt in _s and 'highconf' not in _s:
    _s = _s.replace(_tgt,
        "    import json as _j\n"
        "    try:\n"
        "        _hc = set(_j.load(open('/kaggle/working/highconf.json')))\n"
        "    except Exception:\n"
        "        _hc = set()\n"
        "    for key in ([k for k in sorted(data.keys()) if k not in _hc]\n"
        "                + [k for k in sorted(data.keys()) if k in _hc]):")
    open('starter.py', 'w').write(_s)
    print('starter.py 큐 재배열 패치 완료')
elif 'highconf' in _s:
    print('큐 재배열 패치 이미 적용됨')
else:
    print('!! starter.py 큐 라인 못 찾음 - 재배열 생략 (치명 아님)')
if TEST_KGMON_TASKS is not None:
    _s = open('starter.py').read()
    _s2 = _rem.sub(r'if key not in \[[^\]]*\]:',
                   'if key not in ' + json.dumps(TEST_KGMON_TASKS) + ':', _s)
    if _s2 != _s:
        open('starter.py', 'w').write(_s2)
    print('오프라인 kgmon 대상:', TEST_KGMON_TASKS)

# ---------------- 3) 오프라인 GT 대조 (CPU, 메인 셀) ----------------
if RUN and not _rerun:
    try:
        _SOL = json.load(open(_BASE + 'arc-agi_evaluation_solutions.json'))
    except Exception:
        _SOL = None
    if _SOL:
        n_hit = n_unit = 0
        for key, tid, sp, tc, tag in (OTW + CTW + TWINS):
            for ti_idx, te in enumerate(_CH[tid]['test']):
                fp = f'/kaggle/inference_2step/{key}_{ti_idx}.shape'
                if not os.path.exists(fp):
                    continue
                n_unit += 1
                try:
                    with bz2.BZ2File(fp) as f:
                        outs = pickle.load(f)
                    gt = _SOL[tid][ti_idx] if tid in _SOL else None
                    if gt is not None and any(
                            restore_any(o['solution'], sp,
                                        te['input']).tolist() == gt
                            for o in outs):
                        n_hit += 1
                        print(f'  [적중] {key}_{ti_idx} ({tag})')
                except Exception:
                    pass
        print(f'오프라인: 복원 후 GT 일치 후보 보유 유닛 {n_hit}/{n_unit}')

스위프 모드: 테스트(25분) - U형 600s / 항등 600s
계열 타깃: {'패턴복제': 10, '이동': 10, '배경없음': 10, 'move3': 8} / 총 38태스크
U형 쌍둥이 37개 (26태스크): {'U3c': 3, 'Umono': 21, 'Umono_a': 13} / 항등 82태스크


Loading checkpoint shards: 100%|██████████| 2/2 [00:01<00:00,  1.67it/s]
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


LoRA 병합 253층 <- shapeuni_shu4val.pt
TTT-lite 준비: LoRA r=32, 10스텝/쌍둥이
  TTT 28a6681fu0zzz 16s
  TTT 28a6681fu1zzz 15s
  TTT 31f7f899u0zzz 23s
  TTT 2b83f449u0zzz 22s
  TTT 5545f144u0zzz 28s
  TTT 7666fa5du0zzz 26s
  TTT 7666fa5du1zzz 26s
  TTT faa9f03du0zzz 22s
  TTT faa9f03du1zzz 22s
  TTT 8f3a5a89u0zzz 40s
시간 예산 소진 - 8f3a5a89u1zzz 부터 생략
추론 완료: 10유닛 / 후보 80개 덤프 (614s, TTT=True)
[U형] 페이즈 종료 rc=0 (765s)


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 20.97it/s]


LoRA 병합 253층 <- shapecolor_sc1val.pt
추론 완료: 8유닛 / 후보 60개 덤프 (201s, TTT=False)
[색칠] 페이즈 종료 rc=0 (262s, 8쌍둥이)


...

LoRA 병합 253층 <- nobg_v2val.pt
TTT-lite 준비: LoRA r=32, 10스텝/쌍둥이
  TTT d35bdbdcc0zzz 16s
  TTT f931b4a8c0zzz 24s
  TTT 291dc1e1c0zzz 26s
  TTT 9bbf930dc0zzz 49s
  TTT 21897d95c0zzz 27s
  TTT 800d221bc0zzz 40s
  TTT fc7cae8dc0zzz 50s
  TTT 4c7dc4ddc0zzz 50s
  TTT 6e4f6532c0zzz 63s
시간 예산 소진 - a251c730c0zzz 부터 생략
추론 완료: 11유닛 / 후보 77개 덤프 (1302s, TTT=True)
[계열:배경없음] 페이즈 종료 rc=0 (1360s)


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 21.49it/s]
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


LoRA 병합 253층 <- move3_v2val.pt
TTT-lite 준비: LoRA r=32, 10스텝/쌍둥이
  TTT e8686506c0zzz 11s
  TTT 3dc255dbc0zzz 20s
  TTT d59b0160c0zzz 41s
  TTT 7b3084d4c0zzz 35s
  TTT 5dbc8537c0zzz 24s
  TTT 6ffbe589c0zzz 42s
  TTT 2ba387bcc0zzz 52s
  TTT cbebaa4bc0zzz 53s
추론 완료: 9유닛 / 후보 65개 덤프 (767s, TTT=True)
[계열:move3] 페이즈 종료 rc=0 (840s)


Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00, 21.09it/s]
`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`.


항등 페이즈: 어댑터 병합 없음 (순수 sft139)
TTT-lite 준비: LoRA r=32, 10스텝/쌍둥이
  TTT 20270e3bo0zzz 12s
  TTT 78332cb0o0zzz 16s
  TTT dbff022co0zzz 16s
  TTT eee78d87o0zzz 22s
  TTT 4c3d4a41o0zzz 17s
  TTT b0039139o0zzz 16s
  TTT dd6b8c4bo0zzz 18s
  TTT 7b5033c1o0zzz 15s
  TTT 97d7923eo0zzz 22s
  TTT bf45cf4bo0zzz 23s
  TTT 136b0064o0zzz 18s
  TTT b6f77b65o0zzz 30s
시간 예산 소진 - 1ae2feb7o0zzz 부터 생략
추론 완료: 18유닛 / 후보 141개 덤프 (683s, TTT=True)
[항등] 페이즈 종료 rc=0 (734s)
고신뢰(6/8뷰 일치) 2태스크 -> kgmon 큐 뒤로
starter.py 큐 재배열 패치 완료
오프라인 kgmon 대상: ['0934a4d8', '36a08778', '981571dc', 'aa4ec2a5']
  [적중] 78332cb0o0zzz_0 (orig)
  [적중] 4c3d4a41o0zzz_0 (orig)
  [적중] 4c3d4a41o0zzz_1 (orig)
  [적중] b6f77b65o0zzz_0 (orig)
  [적중] b6f77b65o0zzz_1 (orig)
  [적중] 142ca369c0zzz_0 (cat:패턴복제)
  [적중] 2ba387bcc0zzz_0 (cat:move3)
  [적중] 31f7f899c0zzz_0 (cat:이동)
  [적중] 4c7dc4ddc0zzz_1 (cat:배경없음)
  [적중] 53fb4810c0zzz_0 (cat:패턴복제)
  [적중] d35bdbdcc0zzz_2 (cat:배경없음)
  [적중] e8686506c0zzz_0 (cat:move3)
  [적중] 7666fa5du0zzz_0 (Umono_a)
오프라인: 복원 후 G

In [8]:
!UNSLOTH_DISABLE_STATISTICS=1 TRITON_PTXAS_PATH=/usr/local/cuda/bin/ptxas OMP_NUM_THREADS=6 python starter.py --end-time {global_end_time}

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[Rank 0] start!
[Rank 0] model path: /kaggle/input/qwen3_4b_grids15_sft139/transformers/bfloat16/1
==((====))==  Unsloth 2025.9.7: Fast Qwen3 patching. Transformers: 4.55.4.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.8.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.4.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.32.post2. FA2 = True]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading checkpoint shards: 100%|██████████████████| 2/2 [00:02<00:00,  1.01s/it]
Unsloth: Training embed_tokens in mixed precision to save VRAM
Unsloth: Training lm_head in mixed precision to save VRAM
🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch ever

In [9]:
import os
import json
from arc_loader import ArcDataset
from arc_decoder import ArcDecoder, score_kgmon

rerun_mode = os.getenv("KAGGLE_IS_COMPETITION_RERUN")

if rerun_mode:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
else:
    data = ArcDataset.from_file("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_challenges.json")
    data = data.load_replies("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json")

decoder = ArcDecoder(data.split_multi_replies(), n_guesses=2)

decoder.load_decoded_results("/kaggle/inference_outputs")

submission = data.get_submission(decoder.run_selection_algo(score_kgmon))

with open("submission.json", "w") as f:
    json.dump(submission, f)

if not rerun_mode:
    decoder.benchmark_selection_algos()
    with open("submission.json", "r") as f:
        reload_submission = json.load(f)
    print("*** Reload score:", data.validate_submission(reload_submission))

*** Load solutions from '/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_evaluation_solutions.json'...
*** Generating submission for 5 outputs...
*** Benchmark selection algorithms...
ALL_CORRECT: 0.00048 -  0.00023 30x30 [981571dc_0.rot90.rot90.permute8579032416.ex201.out0]
ALL_CORRECT: 0.00050 -  0.00023 30x30 [981571dc_0.rot90.rot90.permute5386701942.ex213.out0]
ALL_CORRECT: 0.00010 -  0.00023 30x30 [981571dc_0.transpose.rot90.permute4907281356.ex210.out0]
ALL_CORRECT: 0.00005 -  0.00023 30x30 [981571dc_0.permute9302675184.ex302.out0]
ALL_CORRECT: 0.00003 -  0.00023 30x30 [981571dc_0.rot90.permute4950812673.ex013.out0]
ALL_CORRECT: 0.00059 -  0.00023 30x30 [981571dc_0.transpose.rot90.rot90.rot90.permute7690351428.ex231.out0]
ALL_CORRECT: 0.00452 -  0.00023 30x30 [981571dc_0.transpose.permute1602397845.ex013.out0]
ALL_CORRECT: 0.00020 -  0.00023 30x30 [981571dc_0.rot90.rot90.rot90.permute1654278039.ex120.out0]
ALL_CORRECT: 0.00026 -  0.00023 30x30 [981571dc_0.transpose.ro

In [10]:
# ============================================================================
# [F위반 측정 셀 v1] 오프라인 전용 · GPU 0초 · 제출 0회.
#   위치: kgmon 제출 셀(셀10) '다음', 병합 셀(셀12) '이전'.
#         rerun 에서는 자동 skip (측정 전용).
#
#   목적: "kgmon 의 attempt_1 이 고정색(F) 규칙을 위반하는 유닛이 몇 개인가"
#         = 4-(b) 사후 교정기의 잠재 이득 상한을 제출 없이 측정한다.
#
#   지표 (사용자 관례 RESCUED/KILLED):
#     APPLICABLE : F 도출 성공 + 크기 동일 → 검사 가능한 유닛 수
#     VIOLATE    : attempt_1 이 F 칸에서 input 과 다른 유닛 수
#     RESCUED    : 그 교정으로 attempt_1 이 GT 와 일치하게 되는 유닛 (순이득)
#     KILLED     : attempt_1 이 이미 정답인데 F 를 위반 → F 가설이 test 에서
#                  깨진 경우. ★이 값이 0 이 아니면 교정을 켜면 안 된다.
#     NEUTRAL    : 위반이지만 교정해도 여전히 오답 (무해)
#
#   판정: KILLED == 0 이고 RESCUED > 0 일 때만 항목2 패치의 kgmon 확장을 켠다.
# ============================================================================
import os, json
import numpy as np

if os.getenv("KAGGLE_IS_COMPETITION_RERUN"):
    print("rerun 모드: F위반 측정 셀 skip")
else:
    _BASE = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/"
    _CH = json.load(open(_BASE + "arc-agi_evaluation_challenges.json"))
    _SOL = json.load(open(_BASE + "arc-agi_evaluation_solutions.json"))
    _sub = json.load(open("submission.json"))

    def _G(x):
        return np.asarray(x, dtype=int)

    def _bg_of(g):
        """최빈색 = 배경 추정."""
        v, c = np.unique(_G(g), return_counts=True)
        return int(v[c.argmax()])

    def fixed_cols_full(train, include_bg=False):
        """★F 도출 (전체 train 기준, bg 하드코딩 없음).
        F = 모든 train 쌍에서 (input==c) 마스크와 (output==c) 마스크가 동일한 색.
        크기가 다른 쌍이 하나라도 있으면 위치 대응이 없으므로 None."""
        if not train:
            return None
        for p in train:
            if _G(p['input']).shape != _G(p['output']).shape:
                return None
        cols = set()
        for p in train:
            cols |= {int(c) for c in np.unique(_G(p['input']))}
            cols |= {int(c) for c in np.unique(_G(p['output']))}
        # 배경은 기본 제외 (배경 고정은 정보량이 없고 위반 판정만 부풀린다)
        bgs = {_bg_of(p['input']) for p in train}
        bg = bgs.pop() if len(bgs) == 1 else None
        F = []
        for c in sorted(cols):
            if not include_bg and bg is not None and c == bg:
                continue
            ok, present = True, False
            for p in train:
                i, o = _G(p['input']), _G(p['output'])
                if not np.array_equal(i == c, o == c):
                    ok = False
                    break
                if (i == c).any():
                    present = True
            if ok and present:
                F.append(int(c))
        return F

    APPLICABLE = VIOLATE = RESCUED = KILLED = NEUTRAL = 0
    NO_F = SHAPE_MISMATCH = 0
    _killed_list, _rescued_list = [], []
    _viol_ratio = []

    for _tid in sorted(_CH):
        _train = _CH[_tid].get('train', [])
        _F = fixed_cols_full(_train)
        if _F is None:
            SHAPE_MISMATCH += 1
            continue
        if not _F:
            NO_F += 1
            continue
        for _k, _te in enumerate(_CH[_tid]['test']):
            _ti = _G(_te['input'])
            _a1 = _G(_sub[_tid][_k]['attempt_1'])
            if _a1.shape != _ti.shape:
                continue                       # 크기 다른 답 → 위치 대응 불가
            _iF = np.isin(_ti, _F)
            if not _iF.any():
                continue                       # test 에 F 색이 없음
            APPLICABLE += 1
            _bad = int((_a1[_iF] != _ti[_iF]).sum())
            if _bad == 0:
                continue
            VIOLATE += 1
            _viol_ratio.append(_bad / int(_iF.sum()))
            _gt = _SOL[_tid][_k]
            _was_ok = (_a1.tolist() == _gt)
            _fix = _a1.copy()
            _fix[_iF] = _ti[_iF]               # ★F 칸 강제 교정
            _now_ok = (_fix.tolist() == _gt)
            if _was_ok and not _now_ok:
                KILLED += 1
                _killed_list.append(f"{_tid}_{_k}")
            elif not _was_ok and _now_ok:
                RESCUED += 1
                _rescued_list.append(f"{_tid}_{_k}")
            else:
                NEUTRAL += 1

    print("=" * 60)
    print("[F위반 측정]")
    print(f"  F 도출 불가(크기 상이 태스크) : {SHAPE_MISMATCH}")
    print(f"  F 가 빈 태스크               : {NO_F}")
    print(f"  APPLICABLE (검사 가능 유닛)  : {APPLICABLE}")
    print(f"  VIOLATE    (F 위반 유닛)     : {VIOLATE}")
    if _viol_ratio:
        print(f"    위반 비율 중앙값/최대      : "
              f"{np.median(_viol_ratio):.3f} / {max(_viol_ratio):.3f}")
    print(f"  RESCUED    (교정으로 정답)   : {RESCUED}  {_rescued_list[:10]}")
    print(f"  KILLED     (교정으로 오답)   : {KILLED}   {_killed_list[:10]}")
    print(f"  NEUTRAL    (교정해도 오답)   : {NEUTRAL}")
    print("-" * 60)
    if KILLED == 0 and RESCUED > 0:
        print(f"  판정: 안전. kgmon 사후 교정 켜라 (기대 +{RESCUED}유닛)")
    elif KILLED == 0:
        print("  판정: 무해하나 이득 0. attempt_2 슬롯만 낭비 - 켜지 마라")
    else:
        print(f"  판정: ★위험. KILLED={KILLED} - F 가설이 test 에서 깨진다.")
        print("        위반비율 임계(예: 0.2)를 걸어 재측정하라")
    print("=" * 60)

    # ---- 위반 비율 임계별 스윕 (KILLED 를 0 으로 만드는 임계 탐색) ----
    if KILLED > 0:
        print("\n[임계 스윕] 위반비율 <= thr 인 경우만 교정")
        for _thr in (0.05, 0.10, 0.20, 0.30, 0.50):
            r = k = 0
            for _tid in sorted(_CH):
                _F = fixed_cols_full(_CH[_tid].get('train', []))
                if not _F:
                    continue
                for _kk, _te in enumerate(_CH[_tid]['test']):
                    _ti = _G(_te['input'])
                    _a1 = _G(_sub[_tid][_kk]['attempt_1'])
                    if _a1.shape != _ti.shape:
                        continue
                    _iF = np.isin(_ti, _F)
                    if not _iF.any():
                        continue
                    _bad = int((_a1[_iF] != _ti[_iF]).sum())
                    if _bad == 0 or _bad / int(_iF.sum()) > _thr:
                        continue
                    _gt = _SOL[_tid][_kk]
                    _was = (_a1.tolist() == _gt)
                    _fx = _a1.copy(); _fx[_iF] = _ti[_iF]
                    _now = (_fx.tolist() == _gt)
                    if _was and not _now:
                        k += 1
                    elif not _was and _now:
                        r += 1
            print(f"  thr={_thr:.2f} -> RESCUED {r} / KILLED {k}")

[F위반 측정]
  F 도출 불가(크기 상이 태스크) : 39
  F 가 빈 태스크               : 35
  APPLICABLE (검사 가능 유닛)  : 2
  VIOLATE    (F 위반 유닛)     : 0
  RESCUED    (교정으로 정답)   : 0  []
  KILLED     (교정으로 오답)   : 0   []
  NEUTRAL    (교정해도 오답)   : 0
------------------------------------------------------------
  판정: 무해하나 이득 0. attempt_2 슬롯만 낭비 - 켜지 마라


In [11]:
# ============================================================================
# [결합 병합 셀 v2.4.2] 노트북 "맨 마지막" 셀. v2.3 을 통째로 교체.
#   v2.4.1 -> v2.4.2: S형 개수검사를 G1(F+M) 기준·교정 후로 이동
#   ('3색도형' 오폐기 해소).
#   v2.4 -> v2.4.1: S형 '개수 위반' 후보가 페널티 0 으로 통과하던 회귀 수정.
#
#   v2.3 -> v2.4 변경 (전부 플래그로 개별 on/off = 단일변수 검증 가능)
#     ① FORCE_FIX_F      : 항목2. F(고정색) 자리를 '폐기' 대신 '강제 교정'.
#        - 자기검증에서 F 위반 후보를 버리던 것을 위반비율 임계 이하면 교정.
#        - 복원 '후'에도 F 자리를 input 색으로 최종 확정 (마지막 재적용).
#     ② SIL_TIERS        : 항목3+4. 실루엣 매칭 티어.
#        (0,) = v2.3 과 완전 동일 / (0,1) = D4 추가 / (0,1,2) = IoU 폴백까지.
#        폴백 사용분은 페널티로 순위에 반영된다.
#     ③ KGMON_F_REPAIR   : 항목4-b. kgmon attempt_1 이 F 를 위반하면
#        교정본을 attempt_2 후보로. GPU 0초·GT 불필요·attempt_1 불가침 유지.
#        ★반드시 [F위반 측정 셀]에서 KILLED==0 확인 후에만 True 로 둘 것.
#
#   불변 정책 (v2.3 그대로)
#     - attempt_1 불가침. 빈 유닛([[0]])만 우리가 채운다.
#     - attempt_2 배분 = 다양성 순 (U형 분해 > 타깃계열 > 고신뢰 항등)
#     - 원자적 저장 (tmp -> os.replace)
# ============================================================================
import os, json, bz2, pickle
import numpy as np

# ---------------- 플래그 (단일변수 검증용) ----------------
FORCE_FIX_F      = True        # 항목2
F_VIOL_MAX       = 0.20        # F 위반비율 이 값 초과 -> 가설 붕괴로 보고 폐기
SIL_TIERS        = (0,)        # 항목3+4. 1차 제출은 (0,) 로 두고 v2.3 재현 확인
KGMON_F_REPAIR   = False       # 항목4-b. 측정 셀 KILLED==0 확인 후 True
KGMON_F_PRIORITY = 'top'       # 'top' = attempt_2 최우선 / 'bottom' = 최후순위
KGMON_F_VIOL_MAX = 0.20        # kgmon 교정도 같은 임계 적용

print(f"[병합 v2.4] FORCE_FIX_F={FORCE_FIX_F} SIL_TIERS={SIL_TIERS} "
      f"KGMON_F_REPAIR={KGMON_F_REPAIR}")


# ============================================================================
# 실루엣 매칭 v2 (항목3+4) — 스위프 셀과 동일 정의를 유지할 것
# ============================================================================
def _G(x):
    return np.asarray(x, dtype=int)

def _shapes(g, bg, conn=4):
    g = _G(g); H, W = g.shape
    NB = ((1, 0), (-1, 0), (0, 1), (0, -1)) if conn == 4 else \
         tuple((a, b) for a in (-1, 0, 1) for b in (-1, 0, 1) if (a, b) != (0, 0))
    seen = np.zeros((H, W), bool); out = []
    for y in range(H):
        for x in range(W):
            if seen[y, x] or g[y, x] == bg:
                continue
            c0 = int(g[y, x]); st = [(y, x)]; seen[y, x] = True; cs = []
            while st:
                a, b = st.pop(); cs.append((a, b))
                for dy, dx in NB:
                    ny, nx = a + dy, b + dx
                    if 0 <= ny < H and 0 <= nx < W and not seen[ny, nx] \
                            and g[ny, nx] == c0:
                        seen[ny, nx] = True; st.append((ny, nx))
            out.append((c0, cs))
    return out

def _norm(cs):
    y0 = min(y for y, _ in cs); x0 = min(x for _, x in cs)
    return frozenset((y - y0, x - x0) for y, x in cs)

def _center(cs):
    return (sum(y for y, _ in cs) / len(cs), sum(x for _, x in cs) / len(cs))

def _S_silkey(cs):
    return json.dumps(sorted(map(list, _norm(cs))))

def _mask_of(n):
    h = max(y for y, _ in n) + 1; w = max(x for _, x in n) + 1
    a = np.zeros((h, w), bool)
    for y, x in n:
        a[y, x] = True
    return a

def _holes(n):
    """구멍 수 (bbox 패딩 후 바깥 flood fill 로 못 닿는 빈칸 덩어리)."""
    a = _mask_of(n); h, w = a.shape
    m = np.zeros((h + 2, w + 2), bool); m[1:h + 1, 1:w + 1] = a
    NB = ((1, 0), (-1, 0), (0, 1), (0, -1))
    outside = np.zeros_like(m); st = [(0, 0)]; outside[0, 0] = True
    while st:
        y, x = st.pop()
        for dy, dx in NB:
            ny, nx = y + dy, x + dx
            if 0 <= ny < h + 2 and 0 <= nx < w + 2 \
                    and not m[ny, nx] and not outside[ny, nx]:
                outside[ny, nx] = True; st.append((ny, nx))
    holes = 0; vis = np.zeros_like(m)
    for y in range(h + 2):
        for x in range(w + 2):
            if m[y, x] or outside[y, x] or vis[y, x]:
                continue
            holes += 1; st = [(y, x)]; vis[y, x] = True
            while st:
                a2, b2 = st.pop()
                for dy, dx in NB:
                    ny, nx = a2 + dy, b2 + dx
                    if 0 <= ny < h + 2 and 0 <= nx < w + 2 and not m[ny, nx] \
                            and not outside[ny, nx] and not vis[ny, nx]:
                        vis[ny, nx] = True; st.append((ny, nx))
    return holes

def _desc(n):
    """서술자 = (칸수, 구멍수, bbox 높이, bbox 너비)"""
    h = max(y for y, _ in n) + 1; w = max(x for _, x in n) + 1
    return (len(n), _holes(n), h, w)

def _d4key(n):
    """D4 정규형 - 회전/미러된 도형도 같은 키."""
    a = _mask_of(n); best = None
    for k in range(4):
        b = np.rot90(a, k)
        for fl in (False, True):
            c = np.fliplr(b) if fl else b
            key = (c.shape[0], c.shape[1], c.tobytes())
            if best is None or key < best:
                best = key
    return best

def _iou(n1, n2):
    best = len(n1 & n2) / max(len(n1 | n2), 1)
    c1 = _center(list(n1)); c2 = _center(list(n2))
    dy = int(round(c2[0] - c1[0])); dx = int(round(c2[1] - c1[1]))
    if dy or dx:
        s = frozenset((y + dy, x + dx) for y, x in n1)
        best = max(best, len(s & n2) / max(len(s | n2), 1))
    return best

PEN_D4, PEN_APPROX, IOU_MIN, DESC_TOL = 0.15, 0.15, 0.60, 0.30
_TIER_STATS = {}

def _S_sil_paint_v2(p, i, bg, m, colors, tiers=(0,), stats=None):
    """3단 티어 실루엣 매칭. 반환 = 누적 페널티."""
    pen = 0.0
    if not colors:
        return pen
    items = []
    for c, cs in _shapes(i, bg):
        if c in colors:
            n = _norm(cs)
            items.append([c, cs, n, _d4key(n), _desc(n), False])
    if not items:
        return pen
    targets = [cs for c, cs in _shapes(p, bg) if c == m]
    targets.sort(key=len, reverse=True)          # 큰 덩어리 우선 매칭
    for cs in targets:
        n = _norm(cs); d4 = _d4key(n); de = _desc(n); cy, cx = _center(cs)
        pick, tier = None, None
        if 0 in tiers:                                        # 티어0 완전일치
            cand = [it for it in items if not it[5] and it[2] == n]
            if cand:
                pick = cand[0] if len({it[0] for it in cand}) == 1 else \
                    min(cand, key=lambda t: (_center(t[1])[0] - cy) ** 2
                                            + (_center(t[1])[1] - cx) ** 2)
                tier = 0
        if pick is None and 1 in tiers:                       # 티어1 D4
            cand = [it for it in items if not it[5] and it[3] == d4]
            if cand:
                pick = min(cand, key=lambda t: (_center(t[1])[0] - cy) ** 2
                                               + (_center(t[1])[1] - cx) ** 2)
                tier = 1; pen += PEN_D4
        if pick is None and 2 in tiers:                       # 티어2 IoU 폴백
            lo = de[0] * (1 - DESC_TOL); hi = de[0] * (1 + DESC_TOL)
            cand = [it for it in items
                    if not it[5] and lo <= it[4][0] <= hi and it[4][1] == de[1]]
            bv, bi = 0.0, None
            for it in cand:
                v = _iou(n, it[2])
                if v > bv:
                    bv, bi = v, it
            if bi is not None and bv >= IOU_MIN:
                pick = bi; tier = 2
                pen += PEN_APPROX + 0.6 * (1.0 - bv)
        if pick is None:
            continue
        pick[5] = True
        for y, x in cs:
            p[y, x] = pick[0]
        if stats is not None:
            stats[tier] = stats.get(tier, 0) + 1
    return pen


# ============================================================================
# 복원 (v2.3 로직 + 티어/페널티 전달 + ★F 최종 재적용)
# ============================================================================
def _S_groups(spec):
    v = spec['변형']; F, M, V = spec['F'], spec['M'], spec['V']
    if v == '2색':
        return [], F + M + V
    if v == '3색':
        return F, M + V
    if v == '3색도형':
        return F + M, V
    return [], M + V

def _apply_S_grid(g, spec):
    g = _G(g).copy()
    G1, G2 = _S_groups(spec)
    if G1:
        g[np.isin(g, G1)] = spec['m1']
    if G2:
        g[np.isin(g, G2)] = spec['m2']
    return g

def _restore_S(pred, spec, orig_in, tiers=(0,), stats=None):
    p = _G(pred).copy(); i = _G(orig_in); bg = spec['bg']
    pen = 0.0
    G1, G2 = _S_groups(spec)
    m1, m2 = spec.get('m1'), spec['m2']
    same = (p.shape == i.shape)
    if m1 is not None and same:
        mask = (p == m1) & np.isin(i, spec['F'])
        p[mask] = i[mask]
    if m1 is not None:
        pen += _S_sil_paint_v2(p, i, bg, m1,
                               [c for c in spec['M'] if c in G1], tiers, stats)
    mode = spec['변동복원']
    if mode != '모양색':
        pen += _S_sil_paint_v2(p, i, bg, m2, G2, tiers, stats)
    if mode == '모양색':
        f = spec['모양표']
        for cs in [cs for c, cs in _shapes(p, bg) if c == m2]:
            v = f.get(_S_silkey(cs))
            if v is not None:
                for y, x in cs:
                    p[y, x] = v
    if mode in ('위치', '위치단색') and same:
        mask = (p == m2) & np.isin(i, G2)
        p[mask] = i[mask]
    if mode == '전이' and same:
        f = spec['전이표']
        ys, xs = np.where(p == m2)
        for y, x in zip(ys.tolist(), xs.tolist()):
            v = f.get(str(int(i[y, x])))
            if v is not None:
                p[y, x] = v
    if mode in ('단색', '위치단색') and spec.get('단색') is not None:
        p[p == m2] = spec['단색']
    # ★항목2: 고정 최종 재적용. F 칸은 왕복 게이트가 보증한 자리이므로
    #   모델 예측보다 input 이 신뢰도가 높다. 단색 폴백이 덮어쓴 것도 여기서 회수.
    if FORCE_FIX_F and same and spec.get('F'):
        iF = np.isin(i, spec['F'])
        p[iF] = i[iF]
    return p, pen

def _restore_U(pred, spec, orig_in, tiers=(0,), stats=None):
    p = _G(pred).copy(); i = _G(orig_in); m = spec['m']
    if p.shape == i.shape and spec['F']:
        mask = (p == m) & np.isin(i, spec['F'])
        p[mask] = i[mask]
    pen = _S_sil_paint_v2(p, i, 0, m,
                          sorted({int(c) for c in np.unique(i) if c != 0}),
                          tiers, stats)
    # ★항목2: 고정 최종 재적용
    if FORCE_FIX_F and p.shape == i.shape and spec.get('F'):
        iF = np.isin(i, spec['F'])
        p[iF] = i[iF]
    return p, pen

def restore_any_pen(pred, spec, orig_in, tiers=(0,), stats=None):
    """반환 (복원격자 ndarray, 폴백페널티 float)"""
    if spec['종류'] in ('I', 'U색'):
        return _G(pred), 0.0
    if spec['종류'] == 'U':
        return _restore_U(pred, spec, orig_in, tiers, stats)
    return _restore_S(pred, spec, orig_in, tiers, stats)

def restore_any(pred, spec, orig_in):
    """하위호환 (그리드만 반환)"""
    return restore_any_pen(pred, spec, orig_in, tiers=(0,))[0]


# ============================================================================
# F 도출 (항목4-b용, 전체 train 기준 · bg 하드코딩 없음)
# ============================================================================
def _bg_of(g):
    v, c = np.unique(_G(g), return_counts=True)
    return int(v[c.argmax()])

def fixed_cols_full(train):
    """F = 모든 train 쌍에서 (i==c) 마스크 == (o==c) 마스크인 색 (배경 제외).
    크기 다른 쌍이 있으면 위치 대응 불가 -> None."""
    if not train:
        return None
    for p in train:
        if _G(p['input']).shape != _G(p['output']).shape:
            return None
    bgs = {_bg_of(p['input']) for p in train}
    bg = bgs.pop() if len(bgs) == 1 else None
    cols = set()
    for p in train:
        cols |= {int(c) for c in np.unique(_G(p['input']))}
        cols |= {int(c) for c in np.unique(_G(p['output']))}
    F = []
    for c in sorted(cols):
        if bg is not None and c == bg:
            continue
        ok, present = True, False
        for p in train:
            i, o = _G(p['input']), _G(p['output'])
            if not np.array_equal(i == c, o == c):
                ok = False; break
            if (i == c).any():
                present = True
        if ok and present:
            F.append(int(c))
    return F


# ============================================================================
# 실행: kgmon submission 위에 우리 후보 병합
# ============================================================================
_rerun = os.getenv("KAGGLE_IS_COMPETITION_RERUN")
_BASE = "/kaggle/input/competitions/arc-prize-2026-arc-agi-2/"
_SRC = _BASE + ("arc-agi_test_challenges.json" if _rerun
                else "arc-agi_evaluation_challenges.json")
_CH = json.load(open(_SRC))
try:
    _plan = json.load(open("/kaggle/working/분해_계획.json"))
except Exception:
    _plan = {}
try:
    _HC = set(json.load(open("/kaggle/working/highconf.json")))
except Exception:
    _HC = set()
try:
    _TG = set(json.load(open("/kaggle/working/타깃계열.json")).keys())
except Exception:
    _TG = set()
with open("submission.json") as _f:
    _sub = json.load(_f)

def _load_store(path):
    d = {}
    for _key in (os.listdir(path) if os.path.isdir(path) else []):
        try:
            with bz2.BZ2File(os.path.join(path, _key)) as _f:
                _outs = pickle.load(_f)
        except Exception:
            continue
        _bk = _key.split(".")[0]
        d.setdefault(_bk, {})
        for _i, _sm in enumerate(_outs):
            d[_bk][f"{_key}.out{_i}"] = _sm
    return d
_virt = _load_store("/kaggle/inference_2step")

_n_fill = _n_a2 = _n_forced = _n_dropped = 0

for _tid, _twins in _plan.items():
    if _tid not in _CH or _tid not in _sub:
        continue
    for _k in range(len(_CH[_tid]['test'])):
        _ti = _CH[_tid]['test'][_k]['input']
        _tia = np.asarray(_ti)
        _pool = {}                    # 복원판 -> [득표, 점수합, 그리드, I?, U?]
        for _tw in _twins:
            _v = _virt.get(f"{_tw['k']}_{_k}")
            if not _v:
                continue
            _sp = _tw['spec']
            for _sm in _v.values():
                try:
                    _pg = np.asarray(_sm['solution'])
                    _fpen = 0.0
                    # ---------- 자기검증 (항목2: 폐기 -> 강제교정) ----------
                    if _sp['종류'] == 'U' and _sp.get('F') \
                            and _pg.shape == _tia.shape:
                        _iF = np.isin(_tia, _sp['F'])
                        if _iF.any():
                            _bad = int((_pg[_iF] != _sp['m']).sum())
                            _ratio = _bad / int(_iF.sum())
                            if _bad:
                                if not FORCE_FIX_F or _ratio > F_VIOL_MAX:
                                    _n_dropped += 1
                                    continue          # 가설 붕괴 -> 폐기 (v2.3 동작)
                                _pg = _pg.copy(); _pg[_iF] = _sp['m']
                                _fpen += _ratio       # 교정량만큼 소폭 페널티
                                _n_forced += 1
                    elif _sp['종류'] == 'S' and _sp.get('m1') is not None \
                            and _pg.shape == _tia.shape:
                        _iF = np.isin(_tia, _sp['F'])
                        # ★v2.4.2: F 위치위반 = 교정 가능 / 개수는 G1 기준.
                        #   (v2.4.1 의 F-밖 m1 즉시폐기는 '3색도형'(G1=F+M,
                        #    M 도형 이동 가능)을 오폐기했다. v2.3 원검사
                        #    = 총 m1 개수 == G1 셀수 로 복원, 교정 후 검사.)
                        _bad = int((_pg[_iF] != _sp['m1']).sum()) \
                               if _iF.any() else 0
                        if _bad:
                            _ratio = _bad / int(_iF.sum())
                            if not FORCE_FIX_F or _ratio > F_VIOL_MAX:
                                _n_dropped += 1
                                continue
                            _pg = _pg.copy(); _pg[_iF] = _sp['m1']
                            _fpen += _ratio
                            _n_forced += 1
                        _G1 = _S_groups(_sp)[0]
                        if int((_pg == _sp['m1']).sum()) \
                                != int(np.isin(_tia, _G1).sum()):
                            _n_dropped += 1
                            continue
                    # ---------- 복원 (항목3+4 티어) ----------
                    _rg_a, _pen_t = restore_any_pen(_pg, _sp, _ti,
                                                    tiers=SIL_TIERS,
                                                    stats=_TIER_STATS)
                    _rg = _rg_a.tolist()
                    _pen = _fpen + _pen_t
                    _ra = np.asarray(_rg)
                    if _sp['종류'] == 'U' and int((_ra == _sp['m']).sum()):
                        _pen += 1.0                   # 프록시색 잔존 = 복원 실패
                    elif _sp['종류'] == 'S' and int((_ra == _sp['m2']).sum()):
                        _pen += 1.0
                except Exception:
                    continue
                _h = json.dumps(_rg)
                _e = _pool.setdefault(_h, [0, 0.0, _rg, False, False])
                _e[0] += 1
                _e[1] += float(np.mean(_sm.get('score_aug', [1.0]))) + _pen
                if _sp['종류'] == 'I':
                    _e[3] = True
                else:
                    _e[4] = True
        if not _pool:
            continue
        _ranked = sorted(_pool.values(), key=lambda x: -(x[0] - x[1] / x[0]))
        _slot = _sub[_tid][_k]
        _a1 = _slot.get("attempt_1")
        if _a1 == [[0]]:                       # ① kgmon 빈 유닛 -> 우리 채움
            _slot["attempt_1"] = _ranked[0][2]
            _slot["attempt_2"] = _ranked[1][2] if len(_ranked) > 1 \
                                 else _ranked[0][2]
            _n_fill += 1
            continue
        _cand = None                           # ② U형 ③ 타깃계열 ④ 고신뢰 항등
        _iu = [e for e in _ranked if e[4]]
        if _iu and _iu[0][2] != _a1:
            _cand = _iu[0][2]
        if _cand is None and _tid in _TG and _ranked[0][2] != _a1:
            _cand = _ranked[0][2]
        if _cand is None and _tid in _HC:
            _ihc = [e for e in _ranked if e[3]]
            if _ihc and _ihc[0][2] != _a1:
                _cand = _ihc[0][2]
        if _cand is not None and _cand != _slot.get("attempt_2"):
            _slot["attempt_2"] = _cand
            _n_a2 += 1

print(f"결합 병합: 빈유닛 채움 {_n_fill} / attempt_2 교체 {_n_a2} "
      f"(F강제교정 {_n_forced} / F폐기 {_n_dropped}, "
      f"티어사용 {_TIER_STATS}, 계획 {sum(len(v) for v in _plan.values())}쌍둥이)")


# ============================================================================
# 항목4-b: kgmon attempt_1 의 F 사후 교정 -> attempt_2 후보
#   GPU 0초 · GT 불필요 · attempt_1 불가침.
#   ★[F위반 측정 셀]에서 KILLED==0 확인 후에만 켤 것.
# ============================================================================
_n_kg = 0
if KGMON_F_REPAIR:
    for _tid in sorted(_CH):
        if _tid not in _sub:
            continue
        _F = fixed_cols_full(_CH[_tid].get('train', []))
        if not _F:
            continue
        for _k, _te in enumerate(_CH[_tid]['test']):
            _tia = _G(_te['input'])
            _slot = _sub[_tid][_k]
            _a1raw = _slot.get("attempt_1")
            if _a1raw == [[0]]:
                continue                        # 빈 유닛은 위에서 이미 처리
            _a1 = _G(_a1raw)
            if _a1.shape != _tia.shape:
                continue                        # 크기 다르면 위치 대응 불가
            _iF = np.isin(_tia, _F)
            if not _iF.any():
                continue
            _bad = int((_a1[_iF] != _tia[_iF]).sum())
            if _bad == 0:
                continue                        # 이미 F 준수 -> 할 일 없음
            if _bad / int(_iF.sum()) > KGMON_F_VIOL_MAX:
                continue                        # 위반 과다 -> F 가설 붕괴로 보고 보류
            _fix = _a1.copy(); _fix[_iF] = _tia[_iF]
            _fixl = _fix.tolist()
            if _fixl == _a1raw:
                continue
            _cur2 = _slot.get("attempt_2")
            if _fixl == _cur2:
                continue
            if KGMON_F_PRIORITY == 'top':
                _slot["attempt_2"] = _fixl      # 규칙 검증된 교정 -> 최우선
                _n_kg += 1
            elif _cur2 == _a1raw or _cur2 == [[0]]:
                _slot["attempt_2"] = _fixl      # 빈/중복 슬롯일 때만
                _n_kg += 1
    print(f"kgmon F 사후교정: attempt_2 {_n_kg}유닛 "
          f"(우선순위={KGMON_F_PRIORITY}, 임계={KGMON_F_VIOL_MAX})")


# ============================================================================
# 구조 검증 + 원자적 저장
# ============================================================================
for _tid in _CH:
    assert _tid in _sub and len(_sub[_tid]) == len(_CH[_tid]['test'])
    for _slot in _sub[_tid]:
        for _a in ("attempt_1", "attempt_2"):
            assert isinstance(_slot[_a], list) and _slot[_a] \
                   and isinstance(_slot[_a][0], list)
with open("submission_merge.tmp", "w") as _f:
    json.dump(_sub, _f)
os.replace("submission_merge.tmp", "submission.json")

if not _rerun:
    try:
        _SOL = json.load(open(_BASE + "arc-agi_evaluation_solutions.json"))
        _s = 0.0
        for _tid, _v in _SOL.items():
            for _i, _r in enumerate(_v):
                if any(_sub[_tid][_i][_a] == _r
                       for _a in ("attempt_1", "attempt_2")):
                    _s += 1 / len(_v)
        print(f"오프라인 점수(병합 후): {_s:.2f}")
    except Exception as _e:
        print("오프라인 채점 생략:", _e)

[병합 v2.4] FORCE_FIX_F=True SIL_TIERS=(0,) KGMON_F_REPAIR=False
결합 병합: 빈유닛 채움 58 / attempt_2 교체 0 (F강제교정 0 / F폐기 8, 티어사용 {0: 395}, 계획 203쌍둥이)
오프라인 점수(병합 후): 8.67
